# 02 — Tratamento e qualidade dos dados

O objetivo é preparar a série horária e os ciclos de Iatan 2. O guia CAMD orienta unidades, horas parciais e interpretação do CEMS. O NIST orienta o diagnóstico por boxplot e IQR. A PCA sobre o conjunto ON é descritiva. A PCA adicional ajusta todas as transformações no treino. São análises separadas da estimação da Fase 4.

A massa anual é confrontada com a agregação oficial CAMPD da mesma unidade. As horas de calendário permanecem como no arquivo original. Os cortes de modelagem deste notebook são históricos e serão documentados separadamente dos cortes das fases seguintes.

[Guia CAMD](https://www.epa.gov/system/files/documents/2022-07/CAMD%27s%20Power%20Sector%20Emissions%20Data%20Guide%20-%2007182022.pdf) · [NIST, boxplot](https://www.itl.nist.gov/div898/handbook/eda/section3/boxplot.htm) · [Jolliffe e Cadima](https://doi.org/10.1098/rsta.2015.0202) · [Cerqueira et al.](https://arxiv.org/abs/1905.11744)


# Tratamento de dados — Iatan 2025 · v2.0 (auditoria complementar)

As células originais foram preservadas. As auditorias v2 ao final **não sobrescrevem** os dados originais, não modificam os cortes históricos e não utilizam valores oficiais da EPA não fornecidos. Execute o notebook original por inteiro antes das auditorias. Os resultados da v2 serão gerados em `outputs/reports/auditoria_02_v2_2025`. A PCA adicional é ajustada somente ao treino. A conferência EPA é registrada como **pendente** até que se forneça um total oficial independente, com escopo e unidade compatíveis.

## Fundamentação, fontes e decisões do protocolo

As notas metodológicas vinculam as técnicas empregadas às respectivas publicações ou documentações oficiais. As referências completas aparecem ao final deste notebook. As rotinas de leitura, persistência, verificação de integridade e organização dos resultados são implementações do projeto.

**Escolhas do estudo.** Os critérios de elegibilidade, os estados operacionais, os atributos e as datas das partições são decisões do protocolo. A PCA é exploratória e não reduz os atributos utilizados pelos modelos. Sinalizar um extremo estatístico não determina sua exclusão.

**Atualização documental de 24/10/2026.** Foram acrescentadas explicações, citações e referências. As instruções executáveis e as saídas armazenadas foram preservadas. Esta revisão não corresponde a uma nova execução dos experimentos. A rastreabilidade da verificação computacional permanece nos registros de reprodução.

O [guia de fundamentação metodológica](../docs/fundamentacao_metodologica.md) relaciona as fontes, seu alcance e os controles adotados ao longo dos notebooks.


**Atualização bibliográfica de 29/10/2026.** As referências adicionais foram confrontadas com o texto integral fornecido. Elas foram incorporadas onde sustentam diretamente a motivação operacional, a avaliação temporal ou os limites de transferência. Não houve alteração de resultados nem novo treinamento nesta revisão; quando a nova literatura sugere procedimentos não executados, eles são identificados apenas como limites ou possibilidades de validação futura.


In [ ]:
# Registros do experimento usam caminhos relativos à raiz configurada.
# Versões de bibliotecas e resultados são preservados sem identidade da sessão.
import builtins as _builtins
import json as _json
import os as _os
import re as _re
import warnings as _warnings
from pathlib import Path as _Path

def _project_root():
    for name in ("PROJECT_ROOT", "ROOT33", "ROOT", "PROJECT", "SOURCE", "root"):
        value = globals().get(name)
        if isinstance(value, _Path) and value.is_dir():
            return value.resolve()
    return _Path.cwd()

def portable_path(path, root=None):
    path = _Path(path).expanduser()
    root = _Path(root) if root is not None else _project_root()
    try:
        return path.resolve().relative_to(root.resolve()).as_posix()
    except (ValueError, OSError):
        return "external/" + path.name

def public_record(value):
    if isinstance(value, _Path):
        return portable_path(value)
    if isinstance(value, dict):
        return {k: public_record(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [public_record(v) for v in value]
    if isinstance(value, str):
        root = _project_root()
        for prefix in {str(root), root.as_posix()}:
            if value.startswith(prefix + "/") or value.startswith(prefix + "\\"):
                return value[len(prefix)+1:].replace("\\", "/")
            value = value.replace(prefix + "/", "").replace(prefix + "\\", "")
            value = value.replace(prefix, ".")
        if _re.match(r"^(?:[A-Za-z]:[\\/]|/(?:content|tmp|home|Users|mnt)/)", value):
            return "external/" + value.replace("\\", "/").rstrip("/").rsplit("/", 1)[-1]
        return value
    # Cópias de tabelas exportadas normalizam somente colunas textuais.
    if type(value).__module__.startswith("pandas.") and hasattr(value, "columns"):
        result = value.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(public_record)
        return result
    return value

def _project_print(*values, **kwargs):
    root = _project_root()
    if root.is_dir():
        _os.chdir(root)
    return _builtins.print(*(public_record(v) for v in values), **kwargs)

def _project_dump(value, *args, **kwargs):
    return _json.dump(public_record(value), *args, **kwargs)

def _project_dumps(value, *args, **kwargs):
    return _json.dumps(public_record(value), *args, **kwargs)

def _project_csv(frame, *args, **kwargs):
    return public_record(frame).to_csv(*args, **kwargs)

def _project_write_text(path, text, *args, **kwargs):
    return path.write_text(public_record(text), *args, **kwargs)

def _project_display(*values, **kwargs):
    from IPython.display import display as _display
    return _display(*(public_record(v) for v in values), **kwargs)

# Avisos mantêm o conteúdo sem diretórios temporários ou IDs do kernel.
_warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {public_record(str(message))}\n"
)
_os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
_context = _json.loads(_os.environ.get("MLFLOW_RUN_CONTEXT", "{}"))
_context.update({
    "mlflow.user": "equipe", "mlflow.source.name": "iatan",
    "mlflow.source.type": "LOCAL", "mlflow.source.git.commit": "",
    "mlflow.source.git.branch": "", "mlflow.source.git.repoURL": "",
    "mlflow.source.git.diff": "", "mlflow.source.git.dirty": "",
})
_os.environ["MLFLOW_RUN_CONTEXT"] = _json.dumps(_context)

# ============================================================
# CÉLULA 1 — CARREGAMENTO DOS DADOS DA UNIDADE SELECIONADA
# ============================================================
#
# FASE 2 — TRATAMENTO DOS DADOS
#
# OBJETIVO
# ------------------------------------------------------------
# Carregar automaticamente:
#
#   1. a unidade escolhida pelo Notebook 1;
#   2. os dados horários correspondentes.
#
# A identidade da unidade NÃO é definida neste notebook.
#
# Isso garante independência entre:
#
#   FASE 1 — seleção da unidade
#   FASE 2 — tratamento dos dados
#
# Se outra unidade for selecionada no Notebook 1 e os arquivos
# do pipeline forem atualizados, este notebook passará a
# trabalhar automaticamente com a nova unidade.
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

import os
try:
    from google.colab import drive
except ImportError:
    drive = None
from pathlib import Path

import json
import pandas as pd
import numpy as np


# ============================================================
# 2. MONTAR O GOOGLE DRIVE
# ============================================================

if drive is not None:
    drive.mount("/content/drive")


# ============================================================
# 3. PASTA RAIZ DO PROJETO
# ============================================================

DEFAULT_ROOT = '/content/drive/MyDrive/Ciência de Dados - Iatan - Process Mining e Experimentos/Iatan 2025 - entrega revisada/execucao' if drive is not None else str(Path.cwd())
ROOT = Path(os.environ.get("IATAN_PROJECT_ROOT", DEFAULT_ROOT)).expanduser()
ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. DIRETÓRIOS
# ============================================================

DATA = ROOT / "data"

DATA_RAW = DATA / "raw"

DATA_INTERIM = DATA / "interim"

DATA_PROCESSED = DATA / "processed"


OUTPUT = ROOT / "outputs"

REPORTS = OUTPUT / "reports"

FIGURES = OUTPUT / "figures"

TABLES = OUTPUT / "tables"
for folder in (REPORTS, FIGURES, TABLES):
    folder.mkdir(parents=True, exist_ok=True)


PIPELINE_DIR = (
    ROOT
    /
    "pipeline"
)


# ============================================================
# 5. ARQUIVOS PRODUZIDOS PELO NOTEBOOK 1
# ============================================================

CONFIG_PATH = (
    PIPELINE_DIR
    /
    "selected_unit.json"
)

RAW_DATA_PATH = (
    PIPELINE_DIR
    /
    "selected_unit_hourly_raw.parquet"
)


# ============================================================
# 6. VERIFICAR A EXISTÊNCIA DOS ARQUIVOS
# ============================================================

if not CONFIG_PATH.exists():

    raise FileNotFoundError(
        "\nArquivo selected_unit.json não encontrado.\n"
        "Execute primeiro o Notebook 1 — Seleção da Unidade."
    )


if not RAW_DATA_PATH.exists():

    raise FileNotFoundError(
        "\nArquivo selected_unit_hourly_raw.parquet "
        "não encontrado.\n"
        "Execute primeiro a exportação do Notebook 1."
    )


# ============================================================
# 7. CARREGAR IDENTIFICAÇÃO DA UNIDADE
# ============================================================

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    selected_unit = json.load(f)


# ============================================================
# 8. CARREGAR DADOS HORÁRIOS
# ============================================================

df_raw = pd.read_parquet(
    RAW_DATA_PATH
)


# ============================================================
# 9. PRESERVAR A BASE DE ENTRADA
# ============================================================
#
# df_raw
# -------
# Base recebida da fase anterior.
#
# Não deverá ser alterada.
#
#
# df
# --
# Cópia sobre a qual serão executadas as etapas de tratamento.
# ============================================================

df = df_raw.copy()


# ============================================================
# 10. VERIFICAÇÕES DE INTEGRIDADE
# ============================================================

expected_records = int(
    selected_unit["n_records"]
)


if len(df_raw) != expected_records:

    raise ValueError(
        f"O arquivo deveria conter {expected_records:,} registros, "
        f"mas contém {len(df_raw):,}."
    )


# ------------------------------------------------------------
# Conferir Facility ID
# ------------------------------------------------------------

if "facilityId_norm" in df_raw.columns:

    facilities_found = (
        df_raw["facilityId_norm"]
        .astype(str)
        .unique()
    )

    if not (
        len(facilities_found) == 1
        and
        facilities_found[0]
        ==
        str(selected_unit["facility_id"])
    ):

        raise ValueError(
            "O Facility ID do dataset não corresponde "
            "à unidade indicada em selected_unit.json."
        )


# ------------------------------------------------------------
# Conferir Unit ID
# ------------------------------------------------------------

if "unitId_norm" in df_raw.columns:

    units_found = (
        df_raw["unitId_norm"]
        .astype(str)
        .unique()
    )

    if not (
        len(units_found) == 1
        and
        units_found[0]
        ==
        str(selected_unit["unit_id"])
    ):

        raise ValueError(
            "O Unit ID do dataset não corresponde "
            "à unidade indicada em selected_unit.json."
        )


# ============================================================
# 11. GARANTIR FORMATO DO TIMESTAMP
# ============================================================

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)


# ============================================================
# 12. RESUMO DA BASE CARREGADA
# ============================================================

_project_print("=" * 70)
_project_print("FASE 2 — TRATAMENTO DOS DADOS")
_project_print("=" * 70)

_project_print(
    "\nUnidade recebida da fase de seleção:"
)

_project_print(
    f"{selected_unit['facility_name']} "
    f"— Unit {selected_unit['unit_id']}"
)

_project_print(
    "\nFacility ID:",
    selected_unit["facility_id"]
)

_project_print(
    "Estado:",
    selected_unit["state"]
)

_project_print(
    "Ano:",
    selected_unit["year"]
)

_project_print(
    "\nRegistros:",
    f"{len(df):,}"
)

_project_print(
    "Colunas:",
    len(df.columns)
)

_project_print(
    "\nPeríodo:"
)

_project_print(
    df["timestamp"].min(),
    "→",
    df["timestamp"].max()
)

_project_print(
    "\nIntegridade da interface entre notebooks: OK"
)

_project_print(
    "\ndf_raw = base recebida da fase anterior"
)

_project_print(
    "df     = cópia que será utilizada no tratamento"
)

In [ ]:
# ============================================================
# CÉLULA 2 — AUDITORIA INICIAL DA BASE
# ============================================================
#
# OBJETIVO
# ------------------------------------------------------------
# Antes de qualquer tratamento, verificar a estrutura e a
# integridade dos dados recebidos da fase de seleção.
#
# Nesta célula NÃO serão feitas alterações nos dados.
#
# Serão avaliados:
#
#   1. dimensões da base;
#   2. nomes das variáveis;
#   3. tipos de dados;
#   4. valores ausentes;
#   5. registros duplicados;
#   6. duplicidade de timestamps;
#   7. continuidade temporal;
#   8. coerência entre Operating Time, carga e CO2.
#
# Essa auditoria estabelece o diagnóstico inicial da base antes
# das decisões de tratamento.
# ============================================================


# ============================================================
# 1. DIMENSÕES DA BASE
# ============================================================

n_rows, n_cols = df_raw.shape

_project_print("=" * 70)
_project_print("AUDITORIA INICIAL DA BASE")
_project_print("=" * 70)

_project_print(
    "\nDimensões:"
)

_project_print(
    f"{n_rows:,} linhas × {n_cols} colunas"
)


# ============================================================
# 2. VARIÁVEIS DISPONÍVEIS
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("VARIÁVEIS DISPONÍVEIS")
_project_print("=" * 70)

for i, col in enumerate(
    df_raw.columns,
    start=1
):

    _project_print(
        f"{i:02d}. {col}"
    )


# ============================================================
# 3. TIPOS DE DADOS
# ============================================================

dtype_table = pd.DataFrame({

    "variavel":
        df_raw.columns,

    "tipo":
        [
            str(dtype)
            for dtype in df_raw.dtypes
        ]

})


_project_print("\n" + "=" * 70)
_project_print("TIPOS DE DADOS")
_project_print("=" * 70)

_project_display(
    dtype_table
)


# ============================================================
# 4. VALORES AUSENTES
# ============================================================
#
# Para cada variável:
#
#   - número de valores ausentes;
#   - percentual de valores ausentes.
#
# IMPORTANTE:
# valores ausentes não serão automaticamente considerados
# problemas de qualidade.
#
# Por exemplo, carga e CO2 podem estar ausentes quando a
# unidade está desligada.
# ============================================================

missing_table = pd.DataFrame({

    "variavel":
        df_raw.columns,

    "ausentes":
        [
            df_raw[col]
            .isna()
            .sum()

            for col in df_raw.columns
        ]

})


missing_table[
    "percentual"
] = (

    100
    *
    missing_table["ausentes"]
    /
    len(df_raw)

)


missing_table = (
    missing_table
    .sort_values(
        "percentual",
        ascending=False
    )
    .reset_index(drop=True)
)


_project_print("\n" + "=" * 70)
_project_print("VALORES AUSENTES")
_project_print("=" * 70)

_project_display(
    missing_table.round(3)
)


# ============================================================
# 5. REGISTROS DUPLICADOS
# ============================================================

n_duplicates = int(
    df_raw
    .duplicated()
    .sum()
)


_project_print("\n" + "=" * 70)
_project_print("DUPLICATAS")
_project_print("=" * 70)

_project_print(
    "Registros completamente duplicados:",
    n_duplicates
)


# ============================================================
# 6. DUPLICIDADE DE TIMESTAMP
# ============================================================

timestamp_duplicates = int(
    df_raw["timestamp"]
    .duplicated()
    .sum()
)


_project_print(
    "Timestamps duplicados:",
    timestamp_duplicates
)


# ============================================================
# 7. CONTINUIDADE TEMPORAL
# ============================================================
#
# Como o estudo utiliza dados horários de um ano completo,
# esperamos uma sequência contínua com intervalo de 1 hora.
# ============================================================

timestamps = (
    pd.to_datetime(
        df_raw["timestamp"],
        errors="coerce"
    )
    .sort_values()
)


invalid_timestamps = int(
    timestamps
    .isna()
    .sum()
)


time_difference = (
    timestamps
    .diff()
)


non_hourly_intervals = int(
    (
        time_difference
        .dropna()
        != pd.Timedelta(hours=1)
    )
    .sum()
)


expected_timestamps = pd.date_range(

    start=f"{selected_unit['year']}-01-01 00:00:00",

    end=f"{selected_unit['year']}-12-31 23:00:00",

    freq="h"
)


observed_timestamps = pd.DatetimeIndex(
    timestamps.dropna()
)


missing_timestamps = (
    expected_timestamps
    .difference(
        observed_timestamps
    )
)


extra_timestamps = (
    observed_timestamps
    .difference(
        expected_timestamps
    )
)


_project_print("\n" + "=" * 70)
_project_print("INTEGRIDADE TEMPORAL")
_project_print("=" * 70)

_project_print(
    "Timestamps inválidos:",
    invalid_timestamps
)

_project_print(
    "Intervalos diferentes de 1 hora:",
    non_hourly_intervals
)

_project_print(
    "Horas esperadas:",
    len(expected_timestamps)
)

_project_print(
    "Horas ausentes na série:",
    len(missing_timestamps)
)

_project_print(
    "Timestamps fora do período esperado:",
    len(extra_timestamps)
)


# ============================================================
# 8. COERÊNCIA ENTRE OPERAÇÃO, CARGA E CO2
# ============================================================
#
# Operating Time:
#
#   = 0  → unidade não operou durante a hora;
#   > 0  → unidade operou durante pelo menos parte da hora.
#
# Aqui queremos verificar se os valores ausentes de carga e CO2
# estão associados aos períodos de desligamento.
# ============================================================

operating_time = pd.to_numeric(
    df_raw["Operating Time"],
    errors="coerce"
)

gross_load = pd.to_numeric(
    df_raw["Gross Load (MW)"],
    errors="coerce"
)

co2_mass = pd.to_numeric(
    df_raw["CO2 Mass (short tons)"],
    errors="coerce"
)


is_on = (
    operating_time
    .fillna(0)
    > 0
)

is_off = (
    ~is_on
)


operating_hours = int(
    is_on.sum()
)

off_hours = int(
    is_off.sum()
)


load_missing_on = int(
    gross_load[
        is_on
    ]
    .isna()
    .sum()
)

load_missing_off = int(
    gross_load[
        is_off
    ]
    .isna()
    .sum()
)


co2_missing_on = int(
    co2_mass[
        is_on
    ]
    .isna()
    .sum()
)

co2_missing_off = int(
    co2_mass[
        is_off
    ]
    .isna()
    .sum()
)


consistency_table = pd.DataFrame({

    "situacao": [
        "Horas operacionais",
        "Horas desligadas",
        "Carga ausente — unidade ligada",
        "Carga ausente — unidade desligada",
        "CO2 ausente — unidade ligada",
        "CO2 ausente — unidade desligada"
    ],

    "quantidade": [
        operating_hours,
        off_hours,
        load_missing_on,
        load_missing_off,
        co2_missing_on,
        co2_missing_off
    ]

})


_project_print("\n" + "=" * 70)
_project_print("COERÊNCIA OPERACIONAL")
_project_print("=" * 70)

_project_display(
    consistency_table
)


# ============================================================
# 9. VERIFICAR FAIXA DO OPERATING TIME
# ============================================================
#
# Operating Time normalmente varia entre 0 e 1:
#
#   0   = não operou durante a hora;
#   1   = operou durante toda a hora;
#   0–1 = operação durante parte da hora.
# ============================================================

operating_min = (
    operating_time.min()
)

operating_max = (
    operating_time.max()
)


outside_operating_range = int(

    (
        (operating_time < 0)
        |
        (operating_time > 1)
    )
    .sum()

)


partial_hours = int(

    (
        (operating_time > 0)
        &
        (operating_time < 1)
    )
    .sum()

)


_project_print("\n" + "=" * 70)
_project_print("OPERATING TIME")
_project_print("=" * 70)

_project_print(
    "Valor mínimo:",
    operating_min
)

_project_print(
    "Valor máximo:",
    operating_max
)

_project_print(
    "Valores fora do intervalo [0, 1]:",
    outside_operating_range
)

_project_print(
    "Horas com operação parcial:",
    partial_hours
)


# ============================================================
# 10. RESUMO AUTOMÁTICO DA AUDITORIA
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("RESUMO DA AUDITORIA")
_project_print("=" * 70)


audit_checks = {

    "8760 registros":
        len(df_raw) == len(expected_timestamps),

    "sem registros duplicados":
        n_duplicates == 0,

    "sem timestamps duplicados":
        timestamp_duplicates == 0,

    "sem timestamps inválidos":
        invalid_timestamps == 0,

    "série horária contínua":
        (
            non_hourly_intervals == 0
            and
            len(missing_timestamps) == 0
        ),

    "Operating Time dentro de [0,1]":
        outside_operating_range == 0,

    "carga disponível durante operação":
        load_missing_on == 0,

    "CO2 disponível durante operação":
        co2_missing_on == 0
}


for check, result in audit_checks.items():

    symbol = (
        "OK"
        if result
        else "VERIFICAR"
    )

    _project_print(
        f"{symbol:10} | {check}"
    )


# ============================================================
# 11. CONCLUSÃO
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("INTERPRETAÇÃO")
_project_print("=" * 70)


if all(
    audit_checks.values()
):

    _project_print(
        """
A base apresenta integridade estrutural adequada para prosseguir
com o tratamento dos dados.

Não foram identificadas falhas que exijam correção antes das
próximas etapas.

Eventuais valores ausentes de carga e CO2 associados às horas
em que a unidade está desligada devem ser tratados como uma
característica da estrutura dos dados, e não automaticamente
como perda de informação.
"""
    )

else:

    _project_print(
        """
A auditoria identificou pelo menos um aspecto que deve ser
avaliado antes de prosseguir com o tratamento dos dados.

Os itens marcados como VERIFICAR devem ser analisados antes de
qualquer imputação, exclusão ou transformação.
"""
    )

### Campos originais e decisões de tratamento

O significado dos campos reportados e dos indicadores de monitoramento é documentado pela EPA (United States, 2022). A classificação operacional, a análise de ausências e as decisões de conservação ou exclusão implementadas nesta célula pertencem ao protocolo do estudo e devem ser verificadas nos registros de auditoria.


In [ ]:
# Referências metodológicas: (United States, 2022).
# Escopo e decisões próprias estão na nota imediatamente anterior; referências completas ao final.
# ============================================================
# CÉLULA 3 — TRATAMENTO DOS DADOS FALTANTES
#             E ESTADO OPERACIONAL BÁSICO
# ============================================================
#
# FUNDAMENTAÇÃO METODOLÓGICA
# ------------------------------------------------------------
# Esta etapa utiliza técnicas presentes nas disciplinas:
#
#   - tratamento de dados faltantes;
#   - data wrangling;
#   - feature engineering.
#
# A auditoria da Célula 2 mostrou que:
#
#   - a série possui todas as 8.760 horas de 2025;
#   - não existem timestamps ausentes;
#   - não existem duplicatas;
#   - carga e CO2 estão disponíveis em todas as horas em que
#     Operating Time > 0;
#   - os valores ausentes de carga e CO2 ocorrem somente quando
#     Operating Time = 0.
#
# Portanto, NÃO há justificativa para aplicar interpolação,
# média, mediana ou outra imputação estatística.
#
# Os NaN observados nas horas desligadas são AUSÊNCIAS
# ESTRUTURAIS, associadas ao fato de a unidade não estar
# operando.
#
# Estratégia adotada:
#
#   1. preservar as variáveis originais;
#   2. identificar explicitamente o estado operacional;
#   3. substituir por zero SOMENTE os valores ausentes de
#      carga e CO2 em horas comprovadamente desligadas;
#   4. nunca preencher automaticamente valores ausentes durante
#      operação;
#   5. manter indicadores que permitam rastrear quais valores
#      eram originalmente ausentes.
#
# Dessa forma, o tratamento permanece transparente,
# reproduzível e fisicamente interpretável.
# ============================================================


# ============================================================
# 1. GARANTIR TIPOS NUMÉRICOS
# ============================================================

df["Operating Time"] = pd.to_numeric(
    df["Operating Time"],
    errors="coerce"
)

df["Gross Load (MW)"] = pd.to_numeric(
    df["Gross Load (MW)"],
    errors="coerce"
)

df["CO2 Mass (short tons)"] = pd.to_numeric(
    df["CO2 Mass (short tons)"],
    errors="coerce"
)


# ============================================================
# 2. PRESERVAR A INFORMAÇÃO DE AUSÊNCIA ORIGINAL
# ============================================================
#
# Antes de qualquer tratamento, registramos quais observações
# estavam originalmente ausentes.
#
# Isso permite reconstruir posteriormente a situação original
# da base e mantém o tratamento auditável.
# ============================================================

df["gross_load_missing_original"] = (
    df["Gross Load (MW)"]
    .isna()
)

df["co2_missing_original"] = (
    df["CO2 Mass (short tons)"]
    .isna()
)

df["co2_indicator_missing_original"] = (
    df["CO2 Mass Measure Indicator"]
    .isna()
)


# ============================================================
# 3. REPRESENTAÇÃO BÁSICA DO ESTADO OPERACIONAL
# ============================================================
#
# Neste momento ainda NÃO estamos classificando:
#
#   startup
#   ramp-up
#   low load
#   medium load
#   high load
#   ramp-down
#   shutdown
#
# Aqui utilizamos somente a informação diretamente fornecida
# pelo campo Operating Time.
#
# Estados básicos:
#
#   OFF
#       Operating Time = 0
#
#   PARTIAL_OPERATION
#       0 < Operating Time < 1
#
#   FULL_OPERATION
#       Operating Time = 1
#
# Essa classificação não depende de limiares arbitrários.
# ============================================================

operating_time = (
    df["Operating Time"]
)


off_mask = (
    operating_time
    .eq(0)
)


partial_mask = (
    (operating_time > 0)
    &
    (operating_time < 1)
)


full_mask = (
    operating_time
    .eq(1)
)


# ------------------------------------------------------------
# Variáveis binárias auxiliares
# ------------------------------------------------------------

df["is_off"] = (
    off_mask
)

df["is_on"] = (
    operating_time
    .gt(0)
)

df["is_partial_operation"] = (
    partial_mask
)


# ------------------------------------------------------------
# Variável categórica
# ------------------------------------------------------------

df["operational_state_basic"] = np.select(

    [
        off_mask,
        partial_mask,
        full_mask
    ],

    [
        "OFF",
        "PARTIAL_OPERATION",
        "FULL_OPERATION"
    ],

    default="UNDEFINED"
)


# ============================================================
# 4. DIAGNÓSTICO ANTES DO PREENCHIMENTO
# ============================================================
#
# É fundamental separar dois tipos de ausência:
#
# A) ausência estrutural:
#       unidade desligada;
#
# B) ausência durante operação:
#       potencial problema de qualidade.
#
# Apenas o primeiro tipo será tratado automaticamente.
# ============================================================

load_missing_off = int(
    (
        df["Gross Load (MW)"].isna()
        &
        df["is_off"]
    )
    .sum()
)


load_missing_on = int(
    (
        df["Gross Load (MW)"].isna()
        &
        df["is_on"]
    )
    .sum()
)


co2_missing_off = int(
    (
        df["CO2 Mass (short tons)"].isna()
        &
        df["is_off"]
    )
    .sum()
)


co2_missing_on = int(
    (
        df["CO2 Mass (short tons)"].isna()
        &
        df["is_on"]
    )
    .sum()
)


_project_print("=" * 70)
_project_print("DIAGNÓSTICO DOS DADOS FALTANTES")
_project_print("=" * 70)

_project_print(
    "\nCarga ausente em horas OFF:",
    load_missing_off
)

_project_print(
    "Carga ausente em horas ON:",
    load_missing_on
)

_project_print(
    "\nCO2 ausente em horas OFF:",
    co2_missing_off
)

_project_print(
    "CO2 ausente em horas ON:",
    co2_missing_on
)


# ============================================================
# 5. CRIAR VARIÁVEIS TRATADAS
# ============================================================
#
# As colunas originais NÃO são alteradas.
#
# Criamos novas variáveis:
#
#   gross_load_mw_treated
#   co2_mass_short_tons_treated
#
# Regra:
#
#   NaN + unidade OFF → 0
#
# Valores ausentes durante operação, caso existam em outra
# unidade selecionada no futuro, permanecem NaN para impedir
# uma imputação automática metodologicamente inadequada.
# ============================================================

df["gross_load_mw_treated"] = (
    df["Gross Load (MW)"]
    .copy()
)

df["co2_mass_short_tons_treated"] = (
    df["CO2 Mass (short tons)"]
    .copy()
)


# ------------------------------------------------------------
# Carga
# ------------------------------------------------------------

mask = (
    df["is_off"]
    &
    df["gross_load_mw_treated"].isna()
)

df.loc[
    mask,
    "gross_load_mw_treated"
] = 0.0


# ------------------------------------------------------------
# CO2
# ------------------------------------------------------------

mask = (
    df["is_off"]
    &
    df["co2_mass_short_tons_treated"].isna()
)

df.loc[
    mask,
    "co2_mass_short_tons_treated"
] = 0.0


# ============================================================
# 6. INDICADOR DE MEDIÇÃO DE CO2
# ============================================================
#
# Quando a unidade está OFF, não existe medição horária de CO2.
#
# Em vez de deixar a categoria como NaN, criamos uma versão
# tratada indicando explicitamente que a ausência não se aplica
# à operação.
#
# A coluna original também é preservada.
# ============================================================

df["co2_measure_indicator_treated"] = (
    df["CO2 Mass Measure Indicator"]
    .copy()
)


mask = (
    df["is_off"]
    &
    df["co2_measure_indicator_treated"].isna()
)


df.loc[
    mask,
    "co2_measure_indicator_treated"
] = "OFF - NOT APPLICABLE"


# ============================================================
# 7. VERIFICAR RESULTADO DO TRATAMENTO
# ============================================================

remaining_load_missing = int(
    df["gross_load_mw_treated"]
    .isna()
    .sum()
)

remaining_co2_missing = int(
    df["co2_mass_short_tons_treated"]
    .isna()
    .sum()
)


_project_print("\n" + "=" * 70)
_project_print("RESULTADO DO TRATAMENTO")
_project_print("=" * 70)

_project_print(
    "\nValores ausentes restantes — carga tratada:",
    remaining_load_missing
)

_project_print(
    "Valores ausentes restantes — CO2 tratado:",
    remaining_co2_missing
)


# ============================================================
# 8. DISTRIBUIÇÃO DOS ESTADOS OPERACIONAIS BÁSICOS
# ============================================================

operational_state_summary = (

    df[
        "operational_state_basic"
    ]
    .value_counts()
    .rename_axis(
        "estado"
    )
    .reset_index(
        name="horas"
    )

)


operational_state_summary[
    "percentual"
] = (

    100
    *
    operational_state_summary["horas"]
    /
    len(df)

)


_project_print("\n" + "=" * 70)
_project_print("ESTADOS OPERACIONAIS BÁSICOS")
_project_print("=" * 70)

_project_display(
    operational_state_summary
    .round(3)
)


# ============================================================
# 9. TABELA DE DECISÕES DE TRATAMENTO
# ============================================================
#
# Esta tabela documenta explicitamente o que foi feito.
#
# É útil posteriormente para:
#
#   - apresentação;
#   - metodologia do trabalho;
#   - documentação no GitHub.
# ============================================================

treatment_decisions = pd.DataFrame({

    "situacao": [

        "Carga ausente com unidade OFF",

        "CO2 ausente com unidade OFF",

        "Carga ausente com unidade ON",

        "CO2 ausente com unidade ON",

        "Timestamp ausente",

        "Registro duplicado"
    ],

    "quantidade": [

        load_missing_off,

        co2_missing_off,

        load_missing_on,

        co2_missing_on,

        int(
            df["timestamp"]
            .isna()
            .sum()
        ),

        int(
            df
            .duplicated()
            .sum()
        )
    ],

    "tratamento": [

        "Substituição por 0 na variável tratada",

        "Substituição por 0 na variável tratada",

        "Não imputar automaticamente",

        "Não imputar automaticamente",

        "Nenhum tratamento necessário",

        "Nenhum tratamento necessário"
    ],

    "justificativa": [

        "Ausência estrutural associada ao estado OFF",

        "Ausência estrutural associada ao estado OFF",

        "Exigiria investigação antes de qualquer imputação",

        "Exigiria investigação antes de qualquer imputação",

        "Série temporal completa",

        "Não foram identificadas duplicatas"
    ]

})


_project_print("\n" + "=" * 70)
_project_print("DECISÕES DE TRATAMENTO")
_project_print("=" * 70)

_project_display(
    treatment_decisions
)


# ============================================================
# 10. VERIFICAÇÃO PARA MUDANÇA FUTURA DE UNIDADE
# ============================================================
#
# O notebook foi construído para receber qualquer unidade
# selecionada pelo Notebook 1.
#
# Portanto, caso outra unidade apresente valores faltantes
# DURANTE OPERAÇÃO, eles NÃO serão preenchidos silenciosamente.
#
# O notebook sinalizará essa situação para que seja tomada uma
# decisão metodológica específica.
# ============================================================

if (
    load_missing_on > 0
    or
    co2_missing_on > 0
):

    _project_print(
        "\nATENÇÃO:"
    )

    _project_print(
        """
Foram encontrados dados faltantes durante horas operacionais.

Esses valores NÃO foram imputados automaticamente.

Antes de prosseguir, deve-se investigar a estrutura dessas
lacunas e decidir se uma técnica de tratamento de dados
faltantes ensinada na disciplina é metodologicamente adequada.
"""
    )

else:

    _project_print(
        "\nNenhum dado faltante foi identificado durante a operação."
    )

    _project_print(
        "Não há justificativa para interpolação estatística."
    )


# ============================================================
# 11. SALVAR RESULTADO INTERMEDIÁRIO
# ============================================================
#
# O resultado é salvo em data/interim.
#
# Trata-se ainda de uma base intermediária:
# não é o dataset final utilizado no modelo.
# ============================================================

DATA_INTERIM.mkdir(
    parents=True,
    exist_ok=True
)


STEP3_PATH = (
    DATA_INTERIM
    /
    "step03_missing_and_operational_state.parquet"
)


df.to_parquet(
    STEP3_PATH,
    index=False
)


_project_print("\n" + "=" * 70)
_project_print("ETAPA 3 CONCLUÍDA")
_project_print("=" * 70)

_project_print(
    "\nArquivo intermediário:"
)

_project_print(
    STEP3_PATH
)


# ============================================================
# 12. INTERPRETAÇÃO METODOLÓGICA
# ============================================================

_project_print(
    """
A análise dos valores ausentes mostrou que a ausência de dados
de carga e CO2 está associada ao estado desligado da unidade.

Assim, não foi realizada imputação estatística nem interpolação.

Os valores ausentes das horas OFF foram representados como zero
somente em novas variáveis tratadas, enquanto as variáveis
originais foram preservadas.

Também foi criada uma representação básica do comportamento
operacional diretamente a partir de Operating Time:

    OFF
    PARTIAL_OPERATION
    FULL_OPERATION

Essa representação constitui uma etapa inicial de feature
engineering e será posteriormente utilizada na caracterização
mais detalhada do comportamento operacional da unidade.
"""
)

### Regra de intervalo interquartil

Os limites Q1 − 1,5 × IQR e Q3 + 1,5 × IQR, com IQR = Q3 − Q1, correspondem às cercas internas descritas no e-Handbook of Statistical Methods (Guthrie, 2020). Aqui eles sinalizam observações para análise. Não são aplicados como exclusão automática, pois extremos operacionais podem ser dados válidos do processo.


In [ ]:
# Referências metodológicas: (Guthrie, 2020).
# Escopo e decisões próprias estão na nota imediatamente anterior; referências completas ao final.
# ============================================================
# CÉLULA 4 — DETECÇÃO E ANÁLISE DE OUTLIERS
# ============================================================
#
# OBJETIVO
# ------------------------------------------------------------
# Identificar valores extremos nas principais variáveis
# numéricas da unidade:
#
#   - Gross Load (MW)
#   - CO2 Mass (short tons)
#
# A detecção de outliers faz parte das técnicas de
# pré-processamento abordadas na disciplina de Ciência de Dados.
#
# IMPORTANTE:
# ------------------------------------------------------------
# Nesta etapa os outliers serão IDENTIFICADOS, mas NÃO serão
# automaticamente removidos.
#
# Em uma unidade termelétrica, valores extremos podem representar
# fenômenos reais, por exemplo:
#
#   - startup;
#   - shutdown;
#   - rampas rápidas;
#   - operação parcial;
#   - regimes extremos de carga.
#
# Portanto, a decisão de excluir um valor deve considerar seu
# significado operacional e não apenas um critério estatístico.
#
# MÉTODO
# ------------------------------------------------------------
# Será utilizado o intervalo interquartil (IQR):
#
#       IQR = Q3 - Q1
#
# Limite inferior:
#
#       Q1 - 1.5 × IQR
#
# Limite superior:
#
#       Q3 + 1.5 × IQR
#
# A análise será feita apenas para horas em que a unidade
# está operando (Operating Time > 0).
#
# As horas OFF não entram na detecção porque os zeros criados
# anteriormente representam o estado desligado e alterariam
# artificialmente a distribuição.
# ============================================================


import matplotlib.pyplot as plt


# ============================================================
# FUNDAMENTAÇÃO METODOLÓGICA — REGRA DE TUKEY (TUKEY'S FENCES)
# ------------------------------------------------------------
# Os limites Q1 - 1,5 x IQR e Q3 + 1,5 x IQR são conhecidos como
# Tukey's fences (cercas de Tukey), uma regra robusta e
# univariada para sinalizar possíveis valores atípicos.
# Trata-se de uma convenção exploratória, NÃO de uma prova de
# erro, de uma hipótese de normalidade ou de um teste formal.
# A regra é aplicada separadamente a carga e CO2 somente nas
# horas ON; cada flag apenas sinaliza uma observação a examinar.
# A manutenção dos extremos preserva transições reais, rampas
# e demais condições operacionais potencialmente informativas.
# ============================================================


# ============================================================
# 1. SELECIONAR HORAS OPERACIONAIS
# ============================================================

df_on = (
    df[
        df["is_on"]
    ]
    .copy()
)


_project_print("=" * 70)
_project_print("ANÁLISE DE OUTLIERS")
_project_print("=" * 70)

_project_print(
    "\nHoras operacionais analisadas:",
    f"{len(df_on):,}"
)


# ============================================================
# 2. FUNÇÃO PARA CÁLCULO DOS LIMITES PELO IQR
# ============================================================

def calculate_iqr_limits(series):

    clean_series = (
        series
        .dropna()
    )

    q1 = clean_series.quantile(
        0.25
    )

    q3 = clean_series.quantile(
        0.75
    )

    iqr = (
        q3
        -
        q1
    )

    lower_limit = (
        q1
        -
        1.5 * iqr
    )

    upper_limit = (
        q3
        +
        1.5 * iqr
    )

    return {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_limit": lower_limit,
        "upper_limit": upper_limit
    }


# ============================================================
# 3. CARGA
# ============================================================

load_limits = calculate_iqr_limits(
    df_on["Gross Load (MW)"]
)


df["load_outlier_iqr"] = False


load_outlier_mask = (

    df["is_on"]

    &

    (
        (
            df["Gross Load (MW)"]
            <
            load_limits["lower_limit"]
        )

        |

        (
            df["Gross Load (MW)"]
            >
            load_limits["upper_limit"]
        )
    )
)


df.loc[
    load_outlier_mask,
    "load_outlier_iqr"
] = True


# ============================================================
# 4. CO2
# ============================================================

co2_limits = calculate_iqr_limits(
    df_on["CO2 Mass (short tons)"]
)


df["co2_outlier_iqr"] = False


co2_outlier_mask = (

    df["is_on"]

    &

    (
        (
            df["CO2 Mass (short tons)"]
            <
            co2_limits["lower_limit"]
        )

        |

        (
            df["CO2 Mass (short tons)"]
            >
            co2_limits["upper_limit"]
        )
    )
)


df.loc[
    co2_outlier_mask,
    "co2_outlier_iqr"
] = True


# ============================================================
# 5. RESUMO DOS LIMITES
# ============================================================

outlier_limits = pd.DataFrame({

    "variavel": [
        "Gross Load (MW)",
        "CO2 Mass (short tons)"
    ],

    "Q1": [
        load_limits["q1"],
        co2_limits["q1"]
    ],

    "Q3": [
        load_limits["q3"],
        co2_limits["q3"]
    ],

    "IQR": [
        load_limits["iqr"],
        co2_limits["iqr"]
    ],

    "limite_inferior": [
        load_limits["lower_limit"],
        co2_limits["lower_limit"]
    ],

    "limite_superior": [
        load_limits["upper_limit"],
        co2_limits["upper_limit"]
    ]

})


_project_print("\n" + "=" * 70)
_project_print("LIMITES PELO MÉTODO IQR")
_project_print("=" * 70)

_project_display(
    outlier_limits.round(3)
)


# ============================================================
# 6. QUANTIDADE DE OUTLIERS
# ============================================================

n_load_outliers = int(
    df["load_outlier_iqr"]
    .sum()
)

n_co2_outliers = int(
    df["co2_outlier_iqr"]
    .sum()
)


outlier_summary = pd.DataFrame({

    "variavel": [
        "Gross Load (MW)",
        "CO2 Mass (short tons)"
    ],

    "outliers": [
        n_load_outliers,
        n_co2_outliers
    ],

    "percentual_horas_operacionais": [

        100
        *
        n_load_outliers
        /
        len(df_on),

        100
        *
        n_co2_outliers
        /
        len(df_on)
    ]

})


_project_print("\n" + "=" * 70)
_project_print("QUANTIDADE DE OUTLIERS")
_project_print("=" * 70)

_project_display(
    outlier_summary.round(3)
)


# ============================================================
# 7. RELAÇÃO COM O ESTADO OPERACIONAL
# ============================================================
#
# Esta análise é particularmente importante.
#
# Um valor estatisticamente extremo pode coincidir com uma
# hora de operação parcial, o que sugere que ele pode estar
# relacionado a uma transição operacional real.
# ============================================================

outliers_by_state = (

    df[
        df["is_on"]
    ]
    .groupby(
        "operational_state_basic"
    )
    .agg(

        horas=(
            "timestamp",
            "size"
        ),

        outliers_carga=(
            "load_outlier_iqr",
            "sum"
        ),

        outliers_co2=(
            "co2_outlier_iqr",
            "sum"
        )

    )
    .reset_index()
)


outliers_by_state[
    "pct_outlier_carga"
] = (

    100
    *
    outliers_by_state["outliers_carga"]
    /
    outliers_by_state["horas"]

)


outliers_by_state[
    "pct_outlier_co2"
] = (

    100
    *
    outliers_by_state["outliers_co2"]
    /
    outliers_by_state["horas"]

)


_project_print("\n" + "=" * 70)
_project_print("OUTLIERS POR ESTADO OPERACIONAL")
_project_print("=" * 70)

_project_display(
    outliers_by_state.round(3)
)


# ============================================================
# 8. BOXPLOTS
# ============================================================
#
# Visualização simples para apresentação.
#
# Os gráficos mostram a distribuição das variáveis durante as
# horas operacionais.
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)


axes[0].boxplot(
    df_on[
        "Gross Load (MW)"
    ].dropna()
)

axes[0].set_title(
    "Gross Load (MW)"
)

axes[0].set_ylabel(
    "MW"
)


axes[1].boxplot(
    df_on[
        "CO2 Mass (short tons)"
    ].dropna()
)

axes[1].set_title(
    "CO₂ Mass"
)

axes[1].set_ylabel(
    "short tons/h"
)


plt.suptitle(
    "Detecção exploratória de valores extremos"
)

plt.tight_layout()

plt.show()


# ============================================================
# 9. OBSERVAÇÕES MAIS EXTREMAS
# ============================================================
#
# Em vez de remover valores automaticamente, mostramos alguns
# dos maiores valores para permitir inspeção operacional.
# ============================================================

extreme_records = (

    df[
        df["load_outlier_iqr"]
        |
        df["co2_outlier_iqr"]
    ][
        [
            "timestamp",
            "Operating Time",
            "Gross Load (MW)",
            "CO2 Mass (short tons)",
            "operational_state_basic",
            "load_outlier_iqr",
            "co2_outlier_iqr"
        ]
    ]
    .copy()
)


_project_print("\n" + "=" * 70)
_project_print("EXEMPLOS DE OBSERVAÇÕES CLASSIFICADAS COMO OUTLIER")
_project_print("=" * 70)


_project_display(
    extreme_records
    .head(20)
)


# ============================================================
# 10. DECISÃO DE TRATAMENTO
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("DECISÃO METODOLÓGICA")
_project_print("=" * 70)


_project_print(
    """
Os valores extremos foram identificados por meio do intervalo
interquartil (IQR).

Entretanto, nenhum registro foi removido ou substituído nesta
etapa.

Em uma série operacional de uma unidade termelétrica, valores
estatisticamente extremos podem representar condições reais de
operação e transição.

Assim, os indicadores de outlier são preservados como variáveis
auxiliares para análise posterior.

A eventual exclusão de observações somente deverá ocorrer caso
seja identificada evidência de erro ou inconsistência no dado.
"""
)


# ============================================================
# 11. SALVAR RESULTADO INTERMEDIÁRIO
# ============================================================

STEP4_PATH = (
    DATA_INTERIM
    /
    "step04_outlier_analysis.parquet"
)


df.to_parquet(
    STEP4_PATH,
    index=False
)


_project_print(
    "\nArquivo intermediário salvo em:"
)

_project_print(
    STEP4_PATH
)

### Origem da representação operacional

Gu, Sward e Zhang (2023) utilizam variáveis operacionais horárias, incluindo carga, tempo de operação, fator de capacidade e fator de rampa, para estimar emissões de unidades geradoras com dados públicos. Suri, de Chalendar e Azevedo (2025) mostram que rampas, partidas e operação em regimes menos eficientes podem modificar o desempenho emissivo de unidades térmicas. Esses resultados oferecem uma fundamentação mais direta para representar trajetória operacional do que referências genéricas sobre ciclagem.

As defasagens, diferenças, janelas retrospectivas e regras de identificação de partidas implementadas aqui são construções do presente projeto. Seus comprimentos e limiares não são atribuídos aos artigos. As variáveis retrospectivas usam somente observações anteriores; a carga da hora corrente continua sendo uma entrada conhecida do estimador.


In [ ]:
# Referências metodológicas: (Gu; Sward; Zhang, 2023; Suri; de Chalendar; Azevedo, 2025).
# Escopo e decisões próprias estão na nota imediatamente anterior; referências completas ao final.
# ============================================================
# CÉLULA 5 — FEATURE ENGINEERING ORIENTADA AO PROCESSO
# ============================================================
#
# OBJETIVO
# ------------------------------------------------------------
# Criar atributos que representem não apenas o estado atual da
# unidade, mas também sua trajetória operacional recente.
#
# FUNDAMENTAÇÃO
# ------------------------------------------------------------
# Feature engineering faz parte das técnicas de pré-processamento
# abordadas nas disciplinas.
#
# O problema deste trabalho possui natureza temporal e
# operacional.
#
# A hipótese futura de modelagem é:
#
#   informações sobre estados e transições operacionais podem
#   contribuir para estimar CO2 além da informação fornecida
#   apenas pela carga instantânea.
#
#
# IMPORTANTE — EVITAR TARGET LEAKAGE
# ------------------------------------------------------------
# CO2 Mass é a variável-alvo.
#
# Portanto, NÃO serão construídas features baseadas em:
#
#   - CO2 da hora anterior;
#   - médias móveis de CO2;
#   - CO2 futuro;
#   - qualquer transformação derivada da variável-alvo.
#
# As features serão construídas exclusivamente a partir de
# informações operacionais disponíveis.
# ============================================================


# ============================================================
# 1. GARANTIR ORDENAÇÃO TEMPORAL
# ============================================================

df = (
    df
    .sort_values("timestamp")
    .reset_index(drop=True)
)


# ============================================================
# 2. CARGA INSTANTÂNEA
# ============================================================
#
# Esta será a principal variável de referência.
#
# Ela representa o cenário mais simples:
#
#       CO2 = f(carga atual)
#
# ============================================================

df["load_current_mw"] = (
    df["gross_load_mw_treated"]
)


# ============================================================
# FUNDAMENTAÇÃO METODOLÓGICA — LAGGED FEATURES (VARIÁVEIS DEFASADAS)
# ------------------------------------------------------------
# A operação Pandas shift(1) produz uma lag feature, ou seja,
# desloca o valor de uma observação horária para t-1.
# Assim, load_lag_1h(t) = P(t-1), enquanto
# operating_time_lag_1h(t) = OperatingTime(t-1).
# A defasagem representa o histórico observado sem recorrer
# a medições futuras para construir essas variáveis.
# ============================================================


# ============================================================
# 3. CARGA DA HORA ANTERIOR
# ============================================================
#
# P(t-1)
#
# Introduz informação sobre a trajetória imediatamente anterior
# da unidade.
# ============================================================

df["load_lag_1h"] = (
    df["load_current_mw"]
    .shift(1)
)


# ============================================================
# 4. VARIAÇÃO HORÁRIA DA CARGA
# ============================================================
#
# ΔP(t) = P(t) - P(t-1)
#
# Interpretação:
#
#   ΔP > 0  → aumento de carga
#   ΔP < 0  → redução de carga
#   ΔP ≈ 0  → operação aproximadamente estável
#
# Essa feature é importante para representar rampas.
# ============================================================

df["delta_load_1h"] = (
    df["load_current_mw"]
    -
    df["load_lag_1h"]
)


# ============================================================
# 5. MÓDULO DA VARIAÇÃO DE CARGA
# ============================================================
#
# |ΔP|
#
# Mede a intensidade da mudança independentemente da direção.
# ============================================================

df["abs_delta_load_1h"] = (
    df["delta_load_1h"]
    .abs()
)


# ============================================================
# FUNDAMENTAÇÃO METODOLÓGICA — ROLLING-WINDOW FEATURES
# ------------------------------------------------------------
# rolling(window=3, min_periods=1) calcula uma estatística
# em janela móvel (rolling-window feature).
# O shift(1) aplicado ANTES de rolling impede que P(t) seja
# incluída na média destinada a representar apenas o passado:
# média de P(t-1), P(t-2) e P(t-3), quando disponíveis.
# Isso previne look-ahead bias e vazamento temporal nessa feature;
# o valor da própria hora atual é mantido separadamente na
# variável load_current_mw, conforme o desenho experimental.
# ============================================================


# ============================================================
# 6. MÉDIA DA CARGA NAS 3 HORAS ANTERIORES
# ============================================================
#
# Essa variável resume o histórico operacional recente.
#
# IMPORTANTE:
# usamos shift(1) antes da média móvel para que a carga da hora
# atual não seja incluída duas vezes no histórico.
# ============================================================

df["load_mean_previous_3h"] = (

    df["load_current_mw"]
    .shift(1)
    .rolling(
        window=3,
        min_periods=1
    )
    .mean()

)


# ============================================================
# 7. OPERATING TIME DA HORA ANTERIOR
# ============================================================

df["operating_time_lag_1h"] = (
    df["Operating Time"]
    .shift(1)
)


# ============================================================
# FUNDAMENTAÇÃO METODOLÓGICA — STATE-TRANSITION EVENT DETECTION
# ------------------------------------------------------------
# A identificação de startup e shutdown é uma detecção de
# eventos por transição de estado (state-transition event
# detection), baseada em regras determinísticas:
# OFF -> ON indica startup; ON -> OFF indica shutdown.
# Não se trata de classificação por um modelo treinado de IA.
# A primeira hora é mantida sem transição inferida porque
# o estado imediatamente anterior ao início da série é
# desconhecido.
# ============================================================


# ============================================================
# 8. IDENTIFICAR STARTUP E SHUTDOWN
# ============================================================
#
# Startup:
#
#       OFF → ON
#
# Shutdown:
#
#       ON → OFF
#
# A primeira observação do ano NÃO é considerada automaticamente
# uma transição, pois o estado de 31/12/2024 não está disponível.
# ============================================================

previous_on = (
    df["is_on"]
    .shift(1)
)


startup_event = (

    df["is_on"]

    &

    previous_on.eq(False)

)


shutdown_event = (

    df["is_off"]

    &

    previous_on.eq(True)

)


df["startup_event"] = (
    startup_event
)

df["shutdown_event"] = (
    shutdown_event
)


# ============================================================
# 9. DIREÇÃO OPERACIONAL
# ============================================================
#
# Criamos uma classificação simples baseada no sinal de ΔP.
#
# Ainda NÃO estamos definindo formalmente estados como
# LOW LOAD / HIGH LOAD ou utilizando limiares arbitrários.
# ============================================================

conditions = [

    df["is_off"],

    (
        df["is_on"]
        &
        (df["delta_load_1h"] > 0)
    ),

    (
        df["is_on"]
        &
        (df["delta_load_1h"] < 0)
    ),

    (
        df["is_on"]
        &
        (df["delta_load_1h"] == 0)
    )
]


choices = [

    "OFF",

    "LOAD_INCREASE",

    "LOAD_DECREASE",

    "STABLE_LOAD"
]


df["load_direction"] = np.select(

    conditions,

    choices,

    default="UNDEFINED"

)


# ============================================================
# 10. TEMPO DESDE O ÚLTIMO STARTUP
# ============================================================
#
# Esta variável representa há quantas horas a unidade está
# operando desde a última transição OFF → ON.
#
# Pode ser útil futuramente para representar efeitos associados
# ao início de um ciclo operacional.
#
# Durante períodos OFF o valor é zero.
# ============================================================

hours_since_startup = []

counter = 0


for is_on_value, startup_value in zip(

    df["is_on"],
    df["startup_event"]

):

    if not is_on_value:

        counter = 0

        hours_since_startup.append(
            0
        )

    else:

        if startup_value:

            counter = 1

        elif counter > 0:

            counter += 1

        else:

            # A unidade pode já estar ligada no início do ano.
            counter = 0

        hours_since_startup.append(
            counter
        )


df["hours_since_startup"] = (
    hours_since_startup
)


# ============================================================
# 11. VARIÁVEIS TEMPORAIS SIMPLES
# ============================================================
#
# Hora do dia e mês podem capturar padrões temporais de
# operação.
#
# Neste momento elas são criadas apenas para caracterização.
# A decisão de utilizá-las em um modelo será feita depois.
# ============================================================

df["hour_of_day"] = (
    df["timestamp"]
    .dt.hour
)

df["month"] = (
    df["timestamp"]
    .dt.month
)


# ============================================================
# 12. RESUMO DAS FEATURES CRIADAS
# ============================================================

engineered_features = [

    "load_current_mw",

    "load_lag_1h",

    "delta_load_1h",

    "abs_delta_load_1h",

    "load_mean_previous_3h",

    "operating_time_lag_1h",

    "startup_event",

    "shutdown_event",

    "load_direction",

    "hours_since_startup",

    "hour_of_day",

    "month"
]


feature_summary = pd.DataFrame({

    "feature":
        engineered_features,

    "tipo":
        [
            str(df[col].dtype)
            for col in engineered_features
        ],

    "ausentes":
        [
            int(
                df[col]
                .isna()
                .sum()
            )
            for col in engineered_features
        ]

})


_project_print("=" * 70)
_project_print("FEATURE ENGINEERING")
_project_print("=" * 70)

_project_display(
    feature_summary
)


# ============================================================
# 13. RESUMO DAS TRANSIÇÕES
# ============================================================

n_startups = int(
    df["startup_event"]
    .sum()
)

n_shutdowns = int(
    df["shutdown_event"]
    .sum()
)


_project_print("\n" + "=" * 70)
_project_print("TRANSIÇÕES OPERACIONAIS")
_project_print("=" * 70)

_project_print(
    "Startups observados:",
    n_startups
)

_project_print(
    "Shutdowns observados:",
    n_shutdowns
)


# ============================================================
# 14. DISTRIBUIÇÃO DA DIREÇÃO DE CARGA
# ============================================================

load_direction_summary = (

    df[
        "load_direction"
    ]
    .value_counts()
    .rename_axis(
        "direcao"
    )
    .reset_index(
        name="horas"
    )

)


load_direction_summary[
    "percentual"
] = (

    100
    *
    load_direction_summary["horas"]
    /
    len(df)

)


_project_print("\n" + "=" * 70)
_project_print("COMPORTAMENTO DA CARGA")
_project_print("=" * 70)

_project_display(
    load_direction_summary
    .round(3)
)


# ============================================================
# 15. EXEMPLO DE UMA TRANSIÇÃO DE STARTUP
# ============================================================

startup_indices = (
    df.index[
        df["startup_event"]
    ]
    .tolist()
)


if len(startup_indices) > 0:

    example_index = (
        startup_indices[0]
    )

    start = max(
        0,
        example_index - 3
    )

    end = min(
        len(df),
        example_index + 8
    )


    example_columns = [

        "timestamp",

        "Operating Time",

        "load_current_mw",

        "load_lag_1h",

        "delta_load_1h",

        "startup_event",

        "load_direction",

        "hours_since_startup",

        "CO2 Mass (short tons)"
    ]


    _project_print(
        "\n" + "=" * 70
    )

    _project_print(
        "EXEMPLO DE STARTUP"
    )

    _project_print(
        "=" * 70
    )


    _project_display(

        df.loc[
            start:end,
            example_columns
        ]

    )


# ============================================================
# 16. SALVAR RESULTADO INTERMEDIÁRIO
# ============================================================

STEP5_PATH = (

    DATA_INTERIM
    /
    "step05_feature_engineering.parquet"

)


df.to_parquet(

    STEP5_PATH,

    index=False

)


_project_print("\n" + "=" * 70)
_project_print("ETAPA 5 CONCLUÍDA")
_project_print("=" * 70)

_project_print(
    "\nArquivo intermediário:"
)

_project_print(
    STEP5_PATH
)


# ============================================================
# 17. INTERPRETAÇÃO METODOLÓGICA
# ============================================================

_project_print(
    """
A engenharia de atributos transformou a série horária original
em uma representação que incorpora informações sobre a
trajetória operacional da unidade.

Além da carga instantânea, passaram a ser representados:

- carga anterior;
- variação horária da carga;
- intensidade da variação;
- histórico recente de carga;
- transições de startup e shutdown;
- tempo desde o último startup;
- direção da mudança de carga.

Nenhuma variável derivada de CO2 foi utilizada como atributo
preditivo, evitando vazamento de informação da variável-alvo.

Essa estrutura permitirá posteriormente comparar uma abordagem
baseada apenas na carga atual com uma abordagem que incorpora
informações sobre o comportamento operacional do processo.
"""
)

### PCA exploratória e padronização

A PCA constrói componentes ortogonais que sintetizam a variação das variáveis; sua interpretação e suas propriedades são discutidas por Jolliffe e Cadima (2016). A padronização e a implementação computacional estão documentadas no guia da biblioteca (Scikit-learn, 2025). Neste notebook, o ajuste sobre o conjunto exploratório serve para descrição das relações entre atributos. Ele não é reutilizado como transformação dos modelos preditivos.


In [ ]:
# Referências metodológicas: (Jolliffe; Cadima, 2016; Scikit-learn, 2025).
# Escopo e decisões próprias estão na nota imediatamente anterior; referências completas ao final.
# ============================================================
# CÉLULA 6 — PCA EXPLORATÓRIA DAS VARIÁVEIS OPERACIONAIS
# ============================================================
#
# OBJETIVO
# ------------------------------------------------------------
# Aplicar Análise de Componentes Principais (PCA) para
# investigar a estrutura das variáveis criadas por meio de
# feature engineering e avaliar possíveis redundâncias.
#
# A pergunta desta etapa é:
#
#   As variáveis que representam o comportamento operacional
#   carregam informações muito redundantes entre si?
#
# A PCA será utilizada APENAS como ferramenta exploratória.
#
# Nenhuma variável será removida ou substituída automaticamente.
#
#
# IMPORTANTE
# ------------------------------------------------------------
# CO2 Mass NÃO participa da PCA.
#
# O CO2 é a variável-alvo do problema e sua inclusão na redução
# de dimensionalidade das variáveis explicativas provocaria
# vazamento de informação (target leakage).
#
#
# ESCOPO DA PCA
# ------------------------------------------------------------
# A análise será realizada somente nas horas em que a unidade
# está operando.
#
# Motivo:
#
#   Nas horas OFF, muitas variáveis operacionais assumem zero.
#   A inclusão das 2.511 horas desligadas poderia fazer a PCA
#   representar principalmente a diferença ON/OFF, enquanto
#   nosso interesse é investigar a variabilidade do processo
#   DURANTE a operação.
#
#
# VARIÁVEIS CONSIDERADAS
# ------------------------------------------------------------
#
#   load_current_mw
#       carga atual
#
#   load_lag_1h
#       carga da hora anterior
#
#   delta_load_1h
#       variação horária da carga
#
#   abs_delta_load_1h
#       intensidade da variação
#
#   load_mean_previous_3h
#       histórico recente de carga
#
#   operating_time_lag_1h
#       condição operacional anterior
#
#   hours_since_startup
#       duração do ciclo operacional atual
#
#
# Variáveis como mês e hora do dia não entram nesta PCA porque
# representam informação temporal/calendário, e não diretamente
# o estado físico do processo.
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt


# ============================================================
# 2. DEFINIR AS FEATURES OPERACIONAIS
# ============================================================

pca_features = [

    "load_current_mw",

    "load_lag_1h",

    "delta_load_1h",

    "abs_delta_load_1h",

    "load_mean_previous_3h",

    "operating_time_lag_1h",

    "hours_since_startup"

]


# ============================================================
# 3. SELECIONAR SOMENTE HORAS OPERACIONAIS
# ============================================================

pca_data = (

    df.loc[
        df["is_on"],
        pca_features
    ]

    .copy()

)


# ============================================================
# 4. VERIFICAR DADOS AUSENTES
# ============================================================
#
# PCA não aceita valores ausentes.
#
# Não faremos imputação automaticamente.
#
# Caso exista alguma observação incompleta em razão das
# operações de lag, ela será apenas excluída desta análise
# exploratória específica.
# ============================================================

rows_before = len(
    pca_data
)


missing_by_feature = (

    pca_data
    .isna()
    .sum()

)


_project_print("=" * 70)
_project_print("PCA — PREPARAÇÃO DOS DADOS")
_project_print("=" * 70)


_project_print(
    "\nHoras operacionais disponíveis:",
    f"{rows_before:,}"
)


_project_print(
    "\nValores ausentes por feature:"
)

_project_display(
    missing_by_feature
    .rename("ausentes")
    .to_frame()
)


pca_data = (

    pca_data
    .dropna()
    .copy()

)


rows_after = len(
    pca_data
)


_project_print(
    "\nObservações utilizadas na PCA:",
    f"{rows_after:,}"
)

_project_print(
    "Observações excluídas apenas por ausência:",
    rows_before - rows_after
)


# ============================================================
# 5. PADRONIZAÇÃO
# ============================================================
#
# A PCA é sensível à escala das variáveis.
#
# As features possuem unidades e amplitudes muito diferentes:
#
#   MW
#   MW/h
#   horas
#   fração de hora
#
# Portanto, antes da PCA, aplicamos padronização:
#
#                x - média
#       z = -------------------
#              desvio-padrão
#
# Essa etapa corresponde ao mesmo princípio do exemplo da
# disciplina, em que a PCA foi executada com centralização
# e reescala das variáveis.
# ============================================================

# FUNDAMENTAÇÃO METODOLÓGICA — Z-SCORE STANDARDIZATION
# ------------------------------------------------------------
# StandardScaler implementa padronização z-score:
# z = (x - media) / desvio_padrao, por feature.
# A centralização e a reescala evitam que variáveis em
# unidades numericamente maiores dominem a PCA por escala.
# Nesta célula, o ajuste utiliza a amostra da PCA inteira
# porque o procedimento é EXCLUSIVAMENTE EXPLORATÓRIO;
# as componentes não entram no treinamento preditivo.
# Se fossem usadas em modelos supervisionados, scaler e PCA
# deveriam ser ajustados apenas no conjunto de treino.
# ============================================================


scaler_pca = StandardScaler()


X_scaled = scaler_pca.fit_transform(
    pca_data
)


# ============================================================
# 6. PCA
# ============================================================
#
# Inicialmente mantemos todas as componentes.
#
# Assim podemos avaliar quanta variância é explicada por cada
# uma antes de decidir se existe justificativa para redução de
# dimensionalidade.
# ============================================================

pca = PCA()


X_pca = pca.fit_transform(
    X_scaled
)


# ============================================================
# 7. VARIÂNCIA EXPLICADA
# ============================================================

explained_variance = (
    pca.explained_variance_ratio_
)


cumulative_variance = (
    explained_variance
    .cumsum()
)


pca_variance_table = pd.DataFrame({

    "componente": [

        f"PC{i}"

        for i in range(
            1,
            len(explained_variance) + 1
        )

    ],

    "variancia_explicada_pct":

        explained_variance
        * 100,

    "variancia_acumulada_pct":

        cumulative_variance
        * 100

})


_project_print("\n" + "=" * 70)
_project_print("VARIÂNCIA EXPLICADA PELA PCA")
_project_print("=" * 70)


_project_display(
    pca_variance_table
    .round(2)
)


# ============================================================
# 8. QUANTAS COMPONENTES EXPLICAM 90% DA VARIÂNCIA?
# ============================================================
#
# O limiar de 90% é utilizado aqui apenas como referência
# exploratória.
#
# Ele NÃO é uma regra automática para seleção.
# ============================================================

n_components_90 = (

    np.argmax(
        cumulative_variance
        >= 0.90
    )

    + 1

)


_project_print(
    "\nComponentes necessárias para explicar "
    "pelo menos 90% da variância:",
    n_components_90
)

_project_print(
    "Total de features analisadas:",
    len(pca_features)
)


# ============================================================
# 9. SCREE PLOT
# ============================================================
#
# O gráfico mostra:
#
#   - variância explicada por cada componente;
#   - variância acumulada.
#
# Esse é um resultado visual particularmente útil para a
# apresentação do Trabalho 1.
# ============================================================

components = np.arange(
    1,
    len(explained_variance) + 1
)


fig, ax1 = plt.subplots(
    figsize=(10, 5)
)


ax1.bar(

    components,

    explained_variance * 100,

    alpha=0.7,

    label="Variância individual"

)


ax1.set_xlabel(
    "Componente principal"
)

ax1.set_ylabel(
    "Variância explicada (%)"
)

ax1.set_xticks(
    components
)


ax2 = ax1.twinx()


ax2.plot(

    components,

    cumulative_variance * 100,

    marker="o",

    linewidth=2,

    label="Variância acumulada"

)


ax2.axhline(

    90,

    linestyle="--",

    linewidth=1.5,

    label="Referência de 90%"

)


ax2.set_ylabel(
    "Variância acumulada (%)"
)


plt.title(
    "PCA das variáveis operacionais — Scree Plot"
)

plt.tight_layout()

plt.show()


# ============================================================
# 10. LOADINGS DAS COMPONENTES
# ============================================================
#
# Os loadings mostram quanto cada variável original contribui
# para cada componente principal.
#
# Isso ajuda a interpretar fisicamente as componentes.
#
# Por exemplo:
#
# uma componente pode estar fortemente associada ao nível de
# carga, enquanto outra pode representar variações/rampas.
# ============================================================

loadings = pd.DataFrame(

    pca.components_.T,

    index=pca_features,

    columns=[

        f"PC{i}"

        for i in range(
            1,
            len(pca_features) + 1
        )

    ]

)


_project_print("\n" + "=" * 70)
_project_print("LOADINGS DAS COMPONENTES PRINCIPAIS")
_project_print("=" * 70)


_project_display(
    loadings
    .round(3)
)


# ============================================================
# 11. VARIÁVEIS MAIS ASSOCIADAS À PC1 E PC2
# ============================================================

pc1_importance = (

    loadings["PC1"]
    .abs()
    .sort_values(
        ascending=False
    )

)


pc2_importance = (

    loadings["PC2"]
    .abs()
    .sort_values(
        ascending=False
    )

)


_project_print("\n" + "=" * 70)
_project_print("MAIORES CONTRIBUIÇÕES PARA PC1")
_project_print("=" * 70)

_project_display(
    pc1_importance
    .rename(
        "|loading PC1|"
    )
    .to_frame()
)


_project_print("\n" + "=" * 70)
_project_print("MAIORES CONTRIBUIÇÕES PARA PC2")
_project_print("=" * 70)

_project_display(
    pc2_importance
    .rename(
        "|loading PC2|"
    )
    .to_frame()
)


# ============================================================
# 12. REPRESENTAÇÃO DAS OBSERVAÇÕES EM PC1 × PC2
# ============================================================
#
# Este gráfico permite observar se diferentes comportamentos
# operacionais aparecem organizados no espaço das componentes.
#
# Para não poluir o gráfico, usamos apenas uma amostra visual
# se houver muitas observações.
# ============================================================

pca_scores = pd.DataFrame(

    X_pca[:, :2],

    columns=[
        "PC1",
        "PC2"
    ],

    index=pca_data.index

)


pca_scores[
    "operational_state"
] = (

    df.loc[
        pca_scores.index,
        "operational_state_basic"
    ]

)


plt.figure(
    figsize=(9, 6)
)


for state in (

    pca_scores[
        "operational_state"
    ]
    .unique()

):

    subset = (

        pca_scores[
            pca_scores[
                "operational_state"
            ]
            ==
            state
        ]

    )


    plt.scatter(

        subset["PC1"],

        subset["PC2"],

        s=10,

        alpha=0.35,

        label=state

    )


plt.xlabel(
    "PC1"
)

plt.ylabel(
    "PC2"
)

plt.title(
    "Representação das horas operacionais no espaço PCA"
)

plt.legend()

plt.grid(
    alpha=0.2
)

plt.show()


# ============================================================
# 13. DECISÃO EXPLORATÓRIA
# ============================================================
#
# Neste momento NÃO transformamos o dataset principal para PCA.
#
# A decisão depende dos resultados obtidos.
# ============================================================

_project_print("\n" + "=" * 70)
_project_print("INTERPRETAÇÃO PARA O TRABALHO")
_project_print("=" * 70)


_project_print(
    f"""
Foram analisadas {len(pca_features)} variáveis operacionais
por meio de Análise de Componentes Principais (PCA).

São necessárias {n_components_90} componentes para explicar
pelo menos 90% da variância observada.

A PCA foi utilizada nesta etapa como ferramenta exploratória
para avaliar redundância entre as características criadas.

As variáveis originais não foram substituídas pelas componentes
principais neste momento.

Essa decisão preserva a interpretabilidade física das variáveis
operacionais enquanto permite avaliar quantitativamente o
potencial de redução de dimensionalidade.
"""
)


# ============================================================
# 14. SALVAR RESULTADOS DA PCA
# ============================================================

PCA_VARIANCE_PATH = (

    TABLES
    /
    "pca_variance_explained.csv"

)


PCA_LOADINGS_PATH = (

    TABLES
    /
    "pca_loadings.csv"

)


_project_csv(pca_variance_table, 

    PCA_VARIANCE_PATH,

    index=False

)


_project_csv(loadings, 

    PCA_LOADINGS_PATH

)


_project_print(
    "\nResultados salvos em:"
)

_project_print(
    PCA_VARIANCE_PATH
)

_project_print(
    PCA_LOADINGS_PATH
)

### Separação cronológica e normalização

A avaliação de modelos para séries temporais precisa considerar a ordem das observações e o procedimento de estimação do desempenho (Cerqueira; Torgo; Mozetič, 2020). Neste estudo, as datas das partições e o calendário das validações são decisões do protocolo. O teste permanece fora do ajuste de cada comparação, mas seu uso reiterado no desenvolvimento limita a independência da avaliação em relação às decisões da pesquisa (Cawley; Talbot, 2010). A divisão cronológica, isoladamente, não elimina esse limite.

A transformação MinMax e a distinção entre ajustar no treino e apenas transformar os outros conjuntos seguem a documentação da biblioteca (Scikit-learn, 2025). Os extremos são estimados somente no treinamento da partição correspondente. O código mantém esse ajuste para validação e teste; quando o alvo é normalizado, a transformação é revertida antes da avaliação em sua unidade original. Essa precaução não substitui a auditoria das demais fontes possíveis de vazamento.


In [ ]:
# Referências metodológicas: (Cerqueira; Torgo; Mozetič, 2020; Cawley; Talbot, 2010; Scikit-learn, 2025).
# Escopo e decisões próprias estão na nota imediatamente anterior; referências completas ao final.
# ============================================================
# CÉLULA 7 — FECHAMENTO DA PCA E PREPARAÇÃO EXPERIMENTAL
# ============================================================
#
# OBJETIVO
# ------------------------------------------------------------
# Esta etapa possui dois objetivos:
#
#   1. registrar a interpretação e a decisão metodológica
#      resultantes da PCA realizada na etapa anterior;
#
#   2. preparar a base para as futuras etapas de modelagem,
#      realizando a separação temporal dos dados e a
#      normalização das variáveis explicativas.
#
#
# ============================================================
# 1. INTERPRETAÇÃO DOS RESULTADOS DA PCA
# ============================================================
#
# Na etapa anterior foram analisadas sete variáveis operacionais:
#
#   - load_current_mw
#   - load_lag_1h
#   - delta_load_1h
#   - abs_delta_load_1h
#   - load_mean_previous_3h
#   - operating_time_lag_1h
#   - hours_since_startup
#
# Os resultados mostraram:
#
#   PC1  = 44,07% da variância
#   PC2  = 16,51%
#   PC3  = 14,15%
#   PC4  = 13,27%
#   PC5  = 11,66%
#
# Variância acumulada:
#
#   PC1–PC2 = 60,57%
#   PC1–PC3 = 74,72%
#   PC1–PC4 = 87,99%
#   PC1–PC5 = 99,65%
#
# Portanto, foram necessárias CINCO componentes para representar
# pelo menos 90% da variância das sete features analisadas.
#
#
# INTERPRETAÇÃO FÍSICA
# ------------------------------------------------------------
#
# Os loadings indicaram que:
#
# PC1 está fortemente relacionada ao NÍVEL DE CARGA:
#
#   load_lag_1h             ≈ 0,561
#   load_mean_previous_3h   ≈ 0,555
#   load_current_mw         ≈ 0,539
#
# Assim, a primeira componente sintetiza principalmente o nível
# de potência recente da unidade.
#
#
# PC2 está fortemente relacionada à DINÂMICA DA CARGA:
#
#   delta_load_1h       ≈ 0,742
#   abs_delta_load_1h   ≈ 0,611
#
# Portanto, a segunda componente está associada principalmente
# às rampas e variações operacionais.
#
#
# A PCA também identificou uma forte redundância matemática.
#
# A última componente apresentou variância praticamente nula.
#
# Isso é coerente com a relação:
#
#       delta_load_1h =
#       load_current_mw - load_lag_1h
#
# Portanto, parte da redundância detectada não é um problema
# da base, mas consequência da própria engenharia de atributos.
#
#
# DECISÃO METODOLÓGICA
# ------------------------------------------------------------
#
# Apesar de a PCA revelar relações relevantes entre as features,
# a redução dimensional não foi considerada suficientemente
# expressiva para justificar a substituição das variáveis
# operacionais originais pelas componentes principais:
#
#       7 features → 5 componentes para >= 90% da variância
#
# Essa redução seria relativamente pequena e diminuiria a
# interpretabilidade física das variáveis.
#
# Dessa forma:
#
#   A PCA será mantida como ANÁLISE EXPLORATÓRIA.
#
#   As componentes principais NÃO serão utilizadas como
#   substitutas das variáveis originais nesta fase.
#
# Essa decisão permite preservar variáveis fisicamente
# interpretáveis, como carga, rampa e tempo desde startup.
#
# ============================================================


_project_print("=" * 70)
_project_print("CONCLUSÃO DA PCA")
_project_print("=" * 70)

_project_print(
    """
A PCA mostrou que existe redundância entre algumas das variáveis
operacionais, principalmente aquelas relacionadas ao nível de
carga.

Entretanto, cinco das sete componentes foram necessárias para
representar pelo menos 90% da variância.

Assim, a redução dimensional seria relativamente pequena frente
à perda de interpretabilidade física decorrente da substituição
das variáveis originais pelas componentes principais.

A PCA será mantida como ferramenta exploratória, mas as features
operacionais originais serão preservadas para as etapas futuras
de modelagem.

A análise também mostrou uma organização física interessante:

PC1 → nível de carga;
PC2 → dinâmica/rampas da carga.

Além disso, a componente de variância praticamente nula é
compatível com a dependência matemática existente entre carga
atual, carga anterior e variação da carga.
"""
)


# ============================================================
# 2. DEFINIÇÃO DO PROBLEMA EXPERIMENTAL
# ============================================================
#
# A variável que futuramente deverá ser estimada é:
#
#       CO2 Mass (short tons)
#
# Portanto:
#
#       y = emissão horária de CO2
#
#
# As variáveis explicativas serão construídas a partir das
# informações operacionais.
#
# Neste momento serão consideradas somente horas em que a
# unidade está operando.
#
# Motivo:
#
# Durante as horas OFF:
#
#   carga = 0
#   CO2 = 0 na variável tratada
#
# A inclusão dessas horas poderia aumentar artificialmente o
# desempenho de um modelo, pois prever zero durante desligamento
# é um problema muito mais simples do que estimar emissões
# durante a operação.
#
# Assim, o problema principal de modelagem será:
#
#       estimar CO2 DURANTE A OPERAÇÃO da unidade.
#
# ============================================================


model_data = (

    df[
        df["is_on"]
    ]
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)

)


# ============================================================
# 3. FEATURES CANDIDATAS
# ============================================================
#
# Mantemos features fisicamente interpretáveis.
#
# CO2 não aparece entre as variáveis explicativas, evitando
# target leakage.
# ============================================================

model_features = [

    "load_current_mw",

    "load_lag_1h",

    "delta_load_1h",

    "abs_delta_load_1h",

    "load_mean_previous_3h",

    "operating_time_lag_1h",

    "hours_since_startup"

]


target = (
    "CO2 Mass (short tons)"
)


# ============================================================
# 4. REMOVER SOMENTE LINHAS INCOMPLETAS PARA O EXPERIMENTO
# ============================================================

model_columns = (
    ["timestamp"]
    +
    model_features
    +
    [target]
)


model_data = (

    model_data[
        model_columns
    ]
    .dropna()
    .copy()

)


_project_print("\n" + "=" * 70)
_project_print("BASE EXPERIMENTAL")
_project_print("=" * 70)

_project_print(
    "\nObservações operacionais disponíveis:",
    f"{len(model_data):,}"
)

_project_print(
    "Número de features:",
    len(model_features)
)

_project_print(
    "Variável-alvo:",
    target
)


# ============================================================
# 5. DIVISÃO TEMPORAL
# ============================================================
#
# A base representa uma série temporal.
#
# Portanto, NÃO será utilizado train_test_split aleatório.
#
# Uma divisão aleatória permitiria que observações futuras
# participassem do treinamento utilizado para prever observações
# do passado, o que não representa um cenário operacional real.
#
# A separação adotada preserva a ordem cronológica:
#
#   TREINO
#       janeiro → setembro
#
#   VALIDAÇÃO
#       outubro
#
#   TESTE
#       novembro → dezembro
#
# A lógica é:
#
#       passado → treinamento
#
#       futuro → avaliação
#
# ============================================================


# FUNDAMENTAÇÃO METODOLÓGICA — TEMPORAL HOLDOUT
# ------------------------------------------------------------
# O protocolo treino (jan-set), validação (out) e teste
# (nov-dez) configura um temporal holdout: partições fixas,
# disjuntas e cronologicamente ordenadas.
# Treinamento, escolha de modelo e avaliação final devem
# respeitar essa ordem. Não é um split aleatório nem uma
# walk-forward cross-validation de múltiplas origens.
# Esta divisão evita que registros do futuro apareçam no treino.
# ============================================================


train_end = pd.Timestamp(
    f"{selected_unit['year']}-09-30 23:00:00"
)

validation_end = pd.Timestamp(
    f"{selected_unit['year']}-10-31 23:00:00"
)


train_data = (

    model_data[
        model_data["timestamp"]
        <=
        train_end
    ]
    .copy()

)


validation_data = (

    model_data[
        (
            model_data["timestamp"]
            >
            train_end
        )
        &
        (
            model_data["timestamp"]
            <=
            validation_end
        )
    ]
    .copy()

)


test_data = (

    model_data[
        model_data["timestamp"]
        >
        validation_end
    ]
    .copy()

)


# ============================================================
# 6. SEPARAÇÃO ENTRE X E y
# ============================================================

X_train = (
    train_data[
        model_features
    ]
    .copy()
)

y_train = (
    train_data[
        target
    ]
    .copy()
)


X_validation = (
    validation_data[
        model_features
    ]
    .copy()
)

y_validation = (
    validation_data[
        target
    ]
    .copy()
)


X_test = (
    test_data[
        model_features
    ]
    .copy()
)

y_test = (
    test_data[
        target
    ]
    .copy()
)


# ============================================================
# 7. RESUMO DA DIVISÃO TEMPORAL
# ============================================================

split_summary = pd.DataFrame({

    "conjunto": [
        "Treino",
        "Validação",
        "Teste"
    ],

    "inicio": [

        train_data[
            "timestamp"
        ].min(),

        validation_data[
            "timestamp"
        ].min(),

        test_data[
            "timestamp"
        ].min()
    ],

    "fim": [

        train_data[
            "timestamp"
        ].max(),

        validation_data[
            "timestamp"
        ].max(),

        test_data[
            "timestamp"
        ].max()
    ],

    "observacoes": [

        len(train_data),

        len(validation_data),

        len(test_data)
    ]

})


split_summary[
    "percentual"
] = (

    100
    *
    split_summary[
        "observacoes"
    ]
    /
    len(model_data)

)


_project_print("\n" + "=" * 70)
_project_print("DIVISÃO TEMPORAL")
_project_print("=" * 70)

_project_display(
    split_summary.round(2)
)


# ============================================================
# 8. NORMALIZAÇÃO MIN–MAX
# ============================================================
#
# A normalização Min–Max foi uma das técnicas de preparação
# estudadas na disciplina.
#
# Transformação:
#
#             x - xmin
#     x' = ---------------
#            xmax - xmin
#
# Em geral, os valores do conjunto de treinamento passam para
# uma escala aproximadamente entre 0 e 1.
#
#
# IMPORTANTE — EVITAR DATA LEAKAGE
# ------------------------------------------------------------
#
# O scaler é ajustado SOMENTE utilizando o conjunto de treino.
#
#       fit → treino
#
# Depois, o mesmo scaler é utilizado para transformar:
#
#       treino
#       validação
#       teste
#
# Se o scaler fosse ajustado utilizando o ano inteiro, valores
# futuros influenciariam a transformação dos dados de treino.
# ============================================================


from sklearn.preprocessing import MinMaxScaler


# FUNDAMENTAÇÃO METODOLÓGICA — ESCALONAMENTO SEM LEAKAGE
# ------------------------------------------------------------
# Min-Max é distinto da padronização z-score usada na PCA.
# O MinMaxScaler é ajustado SOMENTE com o treinamento e
# aplicado a validação e teste sem novo fit. Isso respeita
# o temporal holdout e evita vazamento de estatísticas
# futuras para o pré-processamento preditivo.
# ============================================================


minmax_scaler = MinMaxScaler()


X_train_scaled = (
    minmax_scaler
    .fit_transform(
        X_train
    )
)


X_validation_scaled = (
    minmax_scaler
    .transform(
        X_validation
    )
)


X_test_scaled = (
    minmax_scaler
    .transform(
        X_test
    )
)


# ============================================================
# 9. CONVERTER NOVAMENTE PARA DATAFRAME
# ============================================================

X_train_scaled = pd.DataFrame(

    X_train_scaled,

    columns=model_features,

    index=X_train.index

)


X_validation_scaled = pd.DataFrame(

    X_validation_scaled,

    columns=model_features,

    index=X_validation.index

)


X_test_scaled = pd.DataFrame(

    X_test_scaled,

    columns=model_features,

    index=X_test.index

)


# ============================================================
# 10. VERIFICAÇÃO DA NORMALIZAÇÃO
# ============================================================

normalization_summary = pd.DataFrame({

    "feature":
        model_features,

    "min_treino_normalizado": [

        X_train_scaled[col]
        .min()

        for col in model_features
    ],

    "max_treino_normalizado": [

        X_train_scaled[col]
        .max()

        for col in model_features
    ]

})


_project_print("\n" + "=" * 70)
_project_print("NORMALIZAÇÃO MIN–MAX")
_project_print("=" * 70)

_project_display(
    normalization_summary.round(4)
)


# ============================================================
# 11. OBSERVAÇÃO SOBRE VALIDAÇÃO E TESTE
# ============================================================
#
# Valores de validação ou teste podem eventualmente ficar
# menores que 0 ou maiores que 1.
#
# Isso NÃO representa erro.
#
# Significa apenas que o período futuro apresentou algum valor
# fora do intervalo observado no conjunto de treinamento.
#
# Isso é justamente o comportamento esperado quando o scaler
# é corretamente ajustado apenas com dados passados.
# ============================================================


validation_outside = int(

    (
        (X_validation_scaled < 0)
        |
        (X_validation_scaled > 1)
    )
    .any(axis=1)
    .sum()

)


test_outside = int(

    (
        (X_test_scaled < 0)
        |
        (X_test_scaled > 1)
    )
    .any(axis=1)
    .sum()

)


_project_print(
    "\nObservações de validação com pelo menos uma feature "
    "fora de [0,1]:",
    validation_outside
)

_project_print(
    "Observações de teste com pelo menos uma feature "
    "fora de [0,1]:",
    test_outside
)


# ============================================================
# 12. SALVAR BASE PREPARADA
# ============================================================
#
# Essa base representa o encerramento do pré-processamento
# necessário para o Trabalho 1.
#
# Ainda NÃO foi treinado nenhum modelo de IA.
# ============================================================

DATA_PROCESSED.mkdir(
    parents=True,
    exist_ok=True
)


MODEL_DATA_PATH = (

    DATA_PROCESSED
    /
    "modeling_dataset_operating_hours.parquet"

)


model_data.to_parquet(

    MODEL_DATA_PATH,

    index=False

)


_project_print("\n" + "=" * 70)
_project_print("BASE PREPARADA PARA MODELAGEM")
_project_print("=" * 70)

_project_print(
    "\nArquivo:"
)

_project_print(
    MODEL_DATA_PATH
)


# ============================================================
# 13. CONCLUSÃO DO PRÉ-PROCESSAMENTO
# ============================================================

_project_print(
    """
O pré-processamento necessário para o Trabalho 1 foi concluído.

O fluxo realizado foi:

1. auditoria da qualidade da base;
2. análise e tratamento dos dados faltantes;
3. identificação de outliers;
4. engenharia de atributos orientada ao processo;
5. PCA exploratória;
6. definição da base experimental;
7. divisão temporal em treino, validação e teste;
8. normalização Min–Max das variáveis explicativas.

A PCA foi aplicada como técnica exploratória de redução de
dimensionalidade, mas suas componentes não substituíram as
features originais, pois cinco das sete componentes foram
necessárias para explicar pelo menos 90% da variância.

A estrutura das componentes mostrou ainda que:

PC1 está predominantemente associada ao nível de carga;

PC2 está predominantemente associada à dinâmica e às rampas
operacionais.

Assim, a análise confirmou que as features criadas representam
aspectos distintos do comportamento da unidade.

A divisão treino/validação/teste respeitou a ordem temporal dos
dados, evitando o uso de informações futuras na preparação do
conjunto de treinamento.

A normalização Min–Max foi ajustada exclusivamente sobre o
conjunto de treino, evitando data leakage.

Com isso, a base encontra-se preparada para uma futura etapa de
modelagem das emissões horárias de CO2.
"""
)

In [ ]:
# ============================================================
# CÉLULA 8 — RELATÓRIO DO TRATAMENTO E PREPARAÇÃO DOS DADOS
# ============================================================
#
# OBJETIVO
# ------------------------------------------------------------
# Consolidar as decisões e resultados obtidos no Notebook 2.
#
# Este relatório documenta:
#
#   - auditoria da base;
#   - tratamento de dados faltantes;
#   - análise de outliers;
#   - engenharia de atributos;
#   - PCA exploratória;
#   - preparação da base experimental;
#   - divisão temporal;
#   - normalização Min–Max.
#
# O objetivo é produzir uma síntese metodológica objetiva
# que possa servir de base para:
#
#   - apresentação do Trabalho 1;
#   - README do GitHub;
#   - metodologia do trabalho;
#   - continuidade futura da modelagem.
# ============================================================


from IPython.display import display, Markdown


# ============================================================
# 1. NÚMEROS PRINCIPAIS
# ============================================================

n_total = len(df)

n_operating = int(
    df["is_on"].sum()
)

n_off = int(
    df["is_off"].sum()
)

n_partial = int(
    df["is_partial_operation"].sum()
)

n_full = int(
    (
        df["operational_state_basic"]
        ==
        "FULL_OPERATION"
    ).sum()
)

n_startups = int(
    df["startup_event"].sum()
)

n_shutdowns = int(
    df["shutdown_event"].sum()
)

n_load_outliers = int(
    df["load_outlier_iqr"].sum()
)

n_co2_outliers = int(
    df["co2_outlier_iqr"].sum()
)


# ============================================================
# 2. RELATÓRIO
# ============================================================

report_treatment = f"""
# Relatório de tratamento e preparação dos dados

## 1. Unidade analisada

A unidade utilizada nesta fase foi carregada automaticamente a
partir do resultado do processo de seleção realizado no Notebook 1.

**Unidade:** {selected_unit['facility_name']} — Unit {selected_unit['unit_id']}
**Facility ID:** {selected_unit['facility_id']}
**Estado:** {selected_unit['state']}
**Ano:** {selected_unit['year']}

O notebook de tratamento não possui dependência explícita da
identidade da usina. Caso outra unidade seja selecionada no
Notebook 1, esta etapa poderá ser executada novamente utilizando
os novos dados.


## 2. Auditoria inicial

A base recebida continha:

- **{n_total:,} registros horários**;
- período completo de 01/01/2025 a 31/12/2025;
- ausência de registros duplicados;
- ausência de timestamps duplicados;
- ausência de timestamps inválidos;
- nenhuma lacuna na sequência horária.

A série apresentou, portanto, integridade estrutural adequada
para o pré-processamento.


## 3. Dados faltantes

Foram identificados valores ausentes em carga e emissão de CO₂.

A investigação mostrou que esses valores ocorriam exclusivamente
nas horas em que a unidade estava desligada.

Foram observadas:

- **{n_operating:,} horas operacionais**;
- **{n_off:,} horas desligadas**;
- **{n_partial:,} horas de operação parcial**;
- **{n_full:,} horas de operação completa**.

Não foram encontrados valores ausentes de carga ou CO₂ durante
as horas operacionais.

Dessa forma, não houve justificativa para utilização de
interpolação ou imputação estatística.

Os valores ausentes associados ao estado OFF foram representados
como zero apenas em novas variáveis tratadas, preservando-se as
variáveis originais para rastreabilidade.


## 4. Análise de outliers

A detecção exploratória de valores extremos foi realizada por
meio do método do intervalo interquartil (IQR).

Foram identificados:

- **{n_load_outliers} outliers de carga**;
- **{n_co2_outliers} outliers de CO₂**.

Esse critério é conhecido como **Tukey's fences (regra de Tukey)**:
limite inferior Q1 − 1,5 × IQR e limite superior Q3 + 1,5 × IQR.
Trata-se de uma triagem estatística exploratória e univariada,
sem equivaler a um teste formal ou à confirmação de erros nos dados.

Nenhuma observação foi automaticamente removida.

A decisão foi baseada no fato de que, em um processo industrial,
um valor estatisticamente extremo pode representar uma condição
operacional real, como rampas, startups, shutdowns ou outros
regimes transitórios.

A exclusão de um ponto somente seria justificada mediante
evidência de erro ou inconsistência no dado.


## 5. Engenharia de atributos

Foi realizada feature engineering para representar não apenas
a condição instantânea da unidade, mas também sua trajetória
operacional.

As principais features criadas foram:

- carga atual;
- carga da hora anterior;
- variação horária da carga;
- módulo da variação da carga;
- média das três horas anteriores;
- Operating Time da hora anterior;
- evento de startup;
- evento de shutdown;
- direção da alteração de carga;
- tempo desde o último startup;
- hora do dia;
- mês.

Do ponto de vista metodológico, `shift(1)` constrói
**lagged features (variáveis defasadas)**, como P(t−1) e o
Operating Time da hora anterior. Já `shift(1).rolling(...)`
constrói uma **rolling-window feature (estatística em janela móvel)**:
a média de carga das três horas anteriores, sem incluir a hora
corrente no histórico. A defasagem anterior à média móvel
reduz o risco de **look-ahead bias (antecipação de informação)**
e de vazamento temporal nessa variável.

A identificação de startups e shutdowns corresponde a
**state-transition event detection (detecção de eventos por
transição de estado)**: as transições OFF → ON e ON → OFF são
reconhecidas por regras determinísticas, sem treinamento de
um classificador. O primeiro registro não gera transição
inferida porque não se conhece o estado da hora anterior.

Foram identificados:

- **{n_startups} startups**;
- **{n_shutdowns} shutdowns**.

Nenhuma variável derivada de CO₂ foi utilizada como feature,
evitando target leakage.

Essa representação permite futuramente comparar um modelo
baseado apenas na carga instantânea com outro que incorpora
informações sobre trajetória e estado operacional.


## 6. PCA exploratória

Foi aplicada Análise de Componentes Principais (PCA), conforme
a técnica de feature extraction estudada na disciplina.

Antes da PCA, as variáveis foram submetidas à
**z-score standardization (padronização por escore z)**,
implementada com `StandardScaler`, de forma que cada variável
é centralizada por sua média e dividida pelo desvio-padrão.
Isso evita predomínio artificial de variáveis por suas escalas.
O ajuste nesta etapa utiliza a amostra exploratória e as
componentes não integram os modelos preditivos; seu eventual
uso supervisionado exigiria ajuste apenas no conjunto de treino.

A PCA utilizou sete variáveis operacionais e foi aplicada
somente às horas em que a unidade estava operando.

A variância explicada foi:

- PC1: **44,07%**
- PC2: **16,51%**
- PC3: **14,15%**
- PC4: **13,27%**
- PC5: **11,66%**

Foram necessárias **5 componentes** para representar pelo menos
90% da variância das sete features analisadas.

Os loadings mostraram uma interpretação física clara:

**PC1 — nível de carga**

As maiores contribuições foram:

- carga da hora anterior;
- média de carga das três horas anteriores;
- carga atual.

**PC2 — dinâmica operacional**

As maiores contribuições foram:

- variação horária da carga;
- intensidade absoluta da variação.

A última componente apresentou variância praticamente nula,
resultado coerente com a dependência matemática:

**ΔP(t) = P(t) − P(t−1)**

Assim, parte da redundância identificada é decorrente da própria
engenharia de atributos.


## 7. Decisão sobre a PCA

Apesar de a PCA identificar redundância entre algumas features,
a redução dimensional não foi considerada suficientemente
expressiva para justificar a substituição das variáveis
originais pelas componentes principais.

A transformação seria aproximadamente:

**7 features → 5 componentes para explicar ≥ 90% da variância**

Como a redução é pequena e as componentes possuem menor
interpretabilidade física, optou-se por preservar as features
originais.

A PCA permanece, portanto, como ferramenta exploratória.


## 8. Definição da base experimental

Para a futura modelagem das emissões de CO₂ foram mantidas
somente as horas em que a unidade estava operando.

A base experimental possui:

**{len(model_data):,} observações operacionais.**

Essa decisão evita que as horas OFF, caracterizadas
essencialmente por carga e emissão iguais a zero, aumentem
artificialmente o desempenho dos modelos.

A variável-alvo definida foi:

**CO2 Mass (short tons)**

As sete features operacionais consideradas nesta fase são:

1. load_current_mw;
2. load_lag_1h;
3. delta_load_1h;
4. abs_delta_load_1h;
5. load_mean_previous_3h;
6. operating_time_lag_1h;
7. hours_since_startup.


## 9. Divisão temporal

Como os dados constituem uma série temporal, não foi utilizada
divisão aleatória entre treino e teste.

Esse protocolo denomina-se **temporal holdout**, isto é,
separação cronológica em conjuntos fixos e não sobrepostos
para treinamento, validação e teste. Não corresponde a uma
validação cruzada temporal walk-forward, que empregaria
múltiplas origens de avaliação.

A ordem cronológica foi preservada:

### Treino

Janeiro a setembro de 2025
**{len(train_data):,} observações**
(**{100 * len(train_data) / len(model_data):.2f}%**)

### Validação

Outubro de 2025
**{len(validation_data):,} observações**
(**{100 * len(validation_data) / len(model_data):.2f}%**)

### Teste

Novembro e dezembro de 2025
**{len(test_data):,} observações**
(**{100 * len(test_data) / len(model_data):.2f}%**)

Os números correspondem exclusivamente às horas operacionais da
unidade, e não ao número total de horas de cada período.

A divisão temporal impede que informações futuras sejam
utilizadas para treinamento de modelos destinados a estimar
observações anteriores.


## 10. Normalização Min–Max

As variáveis explicativas foram preparadas utilizando
normalização Min–Max, técnica abordada na disciplina.

O scaler foi ajustado exclusivamente utilizando o conjunto de
treino.

Posteriormente, o mesmo scaler foi utilizado para transformar
os conjuntos de validação e teste.

Essa estratégia evita data leakage durante o pré-processamento.

Foram identificadas:

- **{validation_outside} observações de validação** com alguma
  feature fora do intervalo [0,1];
- **{test_outside} observações de teste** com alguma feature
  fora do intervalo [0,1].

Esse comportamento não representa erro de normalização.

Valores fora de [0,1] nos períodos futuros indicam apenas que
essas observações apresentaram valores além do intervalo mínimo
ou máximo observado durante o treinamento.


## 10.1. Distinção entre os métodos de escalonamento

A padronização z-score foi aplicada exclusivamente no contexto
da PCA exploratória; a normalização Min–Max foi aplicada à
preparação de entradas para modelagem, com ajuste apenas no
treino. São técnicas diferentes e com finalidades diferentes,
embora ambas modifiquem a escala das variáveis.


## 11. Resultado do pré-processamento

O fluxo de preparação dos dados foi:

**Auditoria dos dados
→ tratamento de valores faltantes
→ análise de outliers
→ feature engineering
→ PCA exploratória
→ seleção da base operacional
→ divisão temporal
→ normalização Min–Max.**

As técnicas foram aplicadas de acordo com a natureza dos dados
e com o objetivo do estudo, evitando aplicação automática de
procedimentos sem justificativa metodológica.

Por exemplo:

- não foi utilizada interpolação porque não existiam lacunas
  durante a operação;
- outliers não foram removidos automaticamente;
- PCA foi analisada, mas não utilizada para substituir as
  features;
- SMOTE não foi aplicado, pois o problema é de regressão e não
  de balanceamento de classes;
- a divisão treino/teste foi adaptada para respeitar a natureza
  temporal dos dados.

## 12. Técnicas de pré-processamento e validação temporal

A preparação dos dados emprega as seguintes técnicas estatísticas,
de engenharia de atributos e de validação temporal:

- **Tukey's fences:** detecção exploratória de outliers usando
  1,5 × IQR, sem exclusão automática de observações;
- **Lagged features:** variáveis defasadas de uma hora via
  `shift(1)`;
- **Rolling-window features:** média móvel das três horas
  anteriores via `shift(1).rolling(...)`;
- **State-transition event detection:** identificação de
  startups e shutdowns pelas passagens OFF/ON e ON/OFF;
- **Z-score standardization:** padronização com `StandardScaler`
  antes da PCA exclusivamente exploratória;
- **Temporal holdout:** divisão cronológica fixa de 2025 em
  treino, validação e teste;
- **Min-Max sem data leakage:** scaler ajustado apenas no treino
  e reutilizado nos períodos seguintes.

Essas técnicas compõem o pré-processamento e a preparação
da base operacional para os experimentos preditivos subsequentes.
Este notebook não treina, por si só, um modelo de IA. A avaliação
de modelos deve respeitar a separação cronológica entre
treinamento, validação e teste.


Com isso, a base encontra-se preparada para a futura etapa de
modelagem das emissões horárias de CO₂.
"""


# ============================================================
# 3. EXIBIR RELATÓRIO
# ============================================================

_project_display(
    Markdown(
        report_treatment
    )
)


# ============================================================
# 4. SALVAR RELATÓRIO
# ============================================================

REPORTS.mkdir(
    parents=True,
    exist_ok=True
)


TREATMENT_REPORT_PATH = (

    REPORTS
    /
    "relatorio_tratamento_dados.md"

)


with open(

    TREATMENT_REPORT_PATH,

    "w",

    encoding="utf-8"

) as f:

    f.write(
        report_treatment
    )


_project_print(
    "\nRelatório salvo em:"
)

_project_print(
    TREATMENT_REPORT_PATH
)

In [ ]:
# ============================================================
# FIGURAS PARA A APRESENTAÇÃO
# Estilo visual coerente com os slides do Trabalho 1
# ============================================================
#
# Esta célula define:
#   - paleta de cores;
#   - tipografia e parâmetros visuais;
#   - pasta de saída;
#   - função para salvar simultaneamente em PNG e SVG.
#
# SVG é interessante para PowerPoint porque preserva qualidade
# vetorial. PNG é mantido para visualização e compatibilidade.
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PASTA DE SAÍDA
# ------------------------------------------------------------

FIGURES = ROOT / "outputs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. PALETA DA APRESENTAÇÃO
# ------------------------------------------------------------

COLOR_BG = "#F4F0EA"          # marfim / fundo dos slides
COLOR_PANEL = "#FAF8F4"       # fundo interno dos gráficos
COLOR_TEXT = "#332F2B"        # carvão
COLOR_MUTED = "#8F8881"       # cinza quente
COLOR_GRID = "#DDD5CC"        # linhas suaves

COLOR_ORANGE = "#D98D3D"      # carga
COLOR_RED = "#B2523A"         # CO2 / destaques
COLOR_RED_LIGHT = "#E8D3CC"
COLOR_ORANGE_LIGHT = "#F1DFC7"
COLOR_BEIGE = "#E7DED4"

# ------------------------------------------------------------
# 3. CONFIGURAÇÃO GLOBAL DO MATPLOTLIB
# ------------------------------------------------------------

plt.rcParams.update({
    "figure.facecolor": COLOR_BG,
    "axes.facecolor": COLOR_PANEL,
    "axes.edgecolor": COLOR_GRID,
    "axes.labelcolor": COLOR_MUTED,
    "axes.titlecolor": COLOR_TEXT,

    "xtick.color": COLOR_MUTED,
    "ytick.color": COLOR_MUTED,

    "text.color": COLOR_TEXT,

    "font.family": "DejaVu Sans",
    "font.size": 11,

    "axes.spines.top": False,
    "axes.spines.right": False,

    "axes.grid": False,
})


# ------------------------------------------------------------
# 4. FUNÇÃO DE SALVAMENTO
# ------------------------------------------------------------

def save_figure(fig, filename):
    """
    Salva cada figura em PNG e SVG.

    PNG:
        boa compatibilidade com PowerPoint.

    SVG:
        formato vetorial, ideal para manter nitidez.
    """

    png_path = FIGURES / f"{filename}.png"
    svg_path = FIGURES / f"{filename}.svg"

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor=COLOR_BG
    )

    fig.savefig(
        svg_path,
        bbox_inches="tight",
        facecolor=COLOR_BG
    )

    _project_print("Figura salva:")
    _project_print(" PNG:", png_path)
    _project_print(" SVG:", svg_path)

In [ ]:
# ============================================================
# FIGURA 1
# COMPORTAMENTO REAL DE CARGA E CO2 AO REDOR DE UM STARTUP
# ============================================================
#
# OBJETIVO:
# Mostrar, com dados reais da Iatan Unit 2, que carga elétrica
# e emissão de CO2 evoluem no tempo, mas não são variáveis
# equivalentes.
#
# Em vez de colocar MW e short tons no mesmo eixo, utilizamos
# dois painéis alinhados temporalmente.
#
# Isso evita uma comparação visual indevida entre grandezas
# físicas com unidades diferentes.
# ============================================================


# ------------------------------------------------------------
# 1. GARANTIR ORDENAÇÃO TEMPORAL
# ------------------------------------------------------------

df_plot = (
    df
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. LOCALIZAR O PRIMEIRO STARTUP OBSERVADO
# ------------------------------------------------------------

startup_rows = df_plot[df_plot["startup_event"] == True]

if startup_rows.empty:
    raise ValueError("Nenhum startup_event foi encontrado.")

startup_time = startup_rows.iloc[0]["timestamp"]


# ------------------------------------------------------------
# 3. DEFINIR JANELA TEMPORAL
#
# 6 horas antes + 30 horas depois.
# ------------------------------------------------------------

window_start = startup_time - pd.Timedelta(hours=6)
window_end   = startup_time + pd.Timedelta(hours=30)

window = df_plot[
    (df_plot["timestamp"] >= window_start)
    &
    (df_plot["timestamp"] <= window_end)
].copy()


# ------------------------------------------------------------
# 4. CRIAR FIGURA
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1,
    figsize=(13, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1]}
)

fig.patch.set_facecolor(COLOR_BG)


# ------------------------------------------------------------
# PAINEL 1 — CARGA
# ------------------------------------------------------------

ax = axes[0]

ax.plot(
    window["timestamp"],
    window["load_current_mw"],
    color=COLOR_ORANGE,
    linewidth=2.7
)

ax.axvline(
    startup_time,
    color=COLOR_RED,
    linewidth=1.5,
    linestyle="--"
)

ax.set_ylabel("Carga [MW]")

ax.text(
    startup_time,
    ax.get_ylim()[1],
    "  startup",
    color=COLOR_RED,
    fontsize=10,
    va="top"
)

ax.set_title(
    "Comportamento horário real da unidade ao redor de um startup",
    loc="left",
    fontsize=17,
    fontweight="normal",
    pad=15
)


# ------------------------------------------------------------
# PAINEL 2 — MASSA HORÁRIA DE CO2
# ------------------------------------------------------------

ax = axes[1]

ax.plot(
    window["timestamp"],
    window["CO2 Mass (short tons)"],
    color=COLOR_RED,
    linewidth=2.7
)

ax.axvline(
    startup_time,
    color=COLOR_RED,
    linewidth=1.5,
    linestyle="--"
)

ax.set_ylabel("CO₂ Mass [short tons]")
ax.set_xlabel("Tempo")


# ------------------------------------------------------------
# 5. FORMATAÇÃO DO EIXO TEMPORAL
# ------------------------------------------------------------

axes[1].xaxis.set_major_locator(
    mdates.HourLocator(interval=4)
)

axes[1].xaxis.set_major_formatter(
    mdates.DateFormatter("%d/%m\n%Hh")
)

for ax in axes:
    ax.spines["left"].set_color(COLOR_GRID)
    ax.spines["bottom"].set_color(COLOR_GRID)
    ax.margins(x=0.01)


# ------------------------------------------------------------
# 6. LEGENDA EXPLICATIVA
# ------------------------------------------------------------

fig.text(
    0.10,
    0.01,
    "Dados reais · Iatan Unit 2 · resolução horária · linha tracejada = início do startup observado",
    fontsize=9,
    color=COLOR_MUTED
)

plt.tight_layout(rect=[0, 0.04, 1, 1])

save_figure(
    fig,
    "01_motivacao_carga_co2_startup_real"
)

plt.show()

In [ ]:
# ============================================================
# FIGURA 2
# EXEMPLO REAL DE UM CICLO OPERACIONAL
# ============================================================
#
# OBJETIVO:
# Mostrar uma sequência observada nos dados entre startup e
# shutdown.
#
# A curva superior apresenta a carga.
#
# A faixa inferior representa as categorias derivadas no
# feature engineering:
#
#   OFF
#   LOAD_INCREASE
#   STABLE_LOAD
#   LOAD_DECREASE
#
# Startups e shutdowns são mostrados como eventos.
# ============================================================


df_cycle = (
    df
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 1. IDENTIFICAR STARTUPS E SHUTDOWNS
# ------------------------------------------------------------

startup_times = (
    df_cycle.loc[df_cycle["startup_event"] == True, "timestamp"]
    .sort_values()
    .tolist()
)

shutdown_times = (
    df_cycle.loc[df_cycle["shutdown_event"] == True, "timestamp"]
    .sort_values()
    .tolist()
)


# ------------------------------------------------------------
# 2. CONSTRUIR CICLOS COMPLETOS
#
# Para cada startup, procuramos o primeiro shutdown posterior.
#
# Preferimos ciclos entre 12 e 120 horas para que o gráfico
# seja visualmente interpretável.
# ------------------------------------------------------------

cycles = []

for st in startup_times:

    future_shutdowns = [
        sd for sd in shutdown_times
        if sd > st
    ]

    if not future_shutdowns:
        continue

    sd = future_shutdowns[0]

    duration_h = (
        sd - st
    ).total_seconds() / 3600

    if 12 <= duration_h <= 120:
        cycles.append((st, sd, duration_h))


if not cycles:
    raise ValueError(
        "Nenhum ciclo completo entre 12 e 120 horas foi encontrado."
    )


# ------------------------------------------------------------
# 3. ESCOLHER UM CICLO REPRESENTATIVO
#
# Evitamos simplesmente escolher o mais curto.
# Selecionamos o ciclo cuja duração está mais próxima da
# mediana das durações encontradas.
# ------------------------------------------------------------

durations = np.array([
    c[2] for c in cycles
])

median_duration = np.median(durations)

selected_cycle = min(
    cycles,
    key=lambda c: abs(c[2] - median_duration)
)

cycle_start, cycle_end, duration_h = selected_cycle


# Adicionar contexto OFF antes e depois
plot_start = cycle_start - pd.Timedelta(hours=4)
plot_end   = cycle_end   + pd.Timedelta(hours=4)

cycle = df_cycle[
    (df_cycle["timestamp"] >= plot_start)
    &
    (df_cycle["timestamp"] <= plot_end)
].copy()


_project_print("Ciclo selecionado:")
_project_print(" Startup :", cycle_start)
_project_print(" Shutdown:", cycle_end)
_project_print(f" Duração : {duration_h:.1f} h")


# ------------------------------------------------------------
# 4. DEFINIR CORES DOS ESTADOS
# ------------------------------------------------------------

state_colors = {
    "OFF": "#D8D2CB",
    "LOAD_INCREASE": "#E7A45A",
    "STABLE_LOAD": "#E8DED3",
    "LOAD_DECREASE": "#C97960",
}


# ------------------------------------------------------------
# 5. FIGURA
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 7),
    sharex=True,
    gridspec_kw={
        "height_ratios": [5, 0.55],
        "hspace": 0.08
    }
)

fig.patch.set_facecolor(COLOR_BG)


# ------------------------------------------------------------
# PAINEL SUPERIOR — CARGA
# ------------------------------------------------------------

ax = axes[0]

ax.plot(
    cycle["timestamp"],
    cycle["load_current_mw"],
    color=COLOR_ORANGE,
    linewidth=2.8
)

ax.axvline(
    cycle_start,
    color=COLOR_RED,
    linestyle="--",
    linewidth=1.5
)

ax.axvline(
    cycle_end,
    color=COLOR_RED,
    linestyle="--",
    linewidth=1.5
)

ax.text(
    cycle_start,
    ax.get_ylim()[1],
    " startup",
    color=COLOR_RED,
    va="top",
    fontsize=10
)

ax.text(
    cycle_end,
    ax.get_ylim()[1],
    " shutdown",
    color=COLOR_RED,
    va="top",
    fontsize=10
)

ax.set_ylabel("Carga [MW]")

ax.set_title(
    "Exemplo real de um ciclo operacional da Iatan Unit 2",
    loc="left",
    fontsize=17,
    pad=15
)


# ------------------------------------------------------------
# PAINEL INFERIOR — FAIXA DE ESTADOS DERIVADOS
# ------------------------------------------------------------

ax_state = axes[1]

for _, row in cycle.iterrows():

    state = row["load_direction"]

    color = state_colors.get(
        state,
        COLOR_BEIGE
    )

    start = row["timestamp"]
    end = start + pd.Timedelta(hours=1)

    ax_state.axvspan(
        start,
        end,
        color=color,
        alpha=1.0,
        linewidth=0
    )

ax_state.set_yticks([])
ax_state.set_ylabel("")


# ------------------------------------------------------------
# 6. FORMATAÇÃO
# ------------------------------------------------------------

axes[1].xaxis.set_major_locator(
    mdates.AutoDateLocator(
        minticks=6,
        maxticks=12
    )
)

axes[1].xaxis.set_major_formatter(
    mdates.DateFormatter("%d/%m\n%Hh")
)

axes[1].set_xlabel("Tempo")


# ------------------------------------------------------------
# 7. LEGENDA DOS ESTADOS
# ------------------------------------------------------------

from matplotlib.patches import Patch

legend_items = [
    Patch(
        facecolor=state_colors["OFF"],
        label="OFF"
    ),
    Patch(
        facecolor=state_colors["LOAD_INCREASE"],
        label="aumento de carga"
    ),
    Patch(
        facecolor=state_colors["STABLE_LOAD"],
        label="carga estável"
    ),
    Patch(
        facecolor=state_colors["LOAD_DECREASE"],
        label="redução de carga"
    )
]

axes[0].legend(
    handles=legend_items,
    loc="upper right",
    frameon=False,
    ncol=2,
    fontsize=9
)

plt.tight_layout()

save_figure(
    fig,
    "02_ciclo_operacional_real"
)

plt.show()

In [ ]:
# ============================================================
# FIGURA 3
# MESMA CARGA APROXIMADA, TRAJETÓRIAS OPOSTAS
# ============================================================
#
# OBJETIVO:
# Encontrar automaticamente duas observações reais em que:
#
#     P_A(t) ≈ P_B(t)
#
# porém:
#
#     ΔP_A(t) > 0
#     ΔP_B(t) < 0
#
# Assim mostramos que a mesma carga instantânea pode ocorrer
# em contextos temporais diferentes.
# ============================================================


df_context = (
    df
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 1. MANTER APENAS HORAS OPERACIONAIS COM DADOS VÁLIDOS
# ------------------------------------------------------------

candidates = df_context[
    (df_context["is_on"] == True)
    &
    (df_context["load_current_mw"].notna())
    &
    (df_context["delta_load_1h"].notna())
].copy()


# ------------------------------------------------------------
# 2. EXIGIR RAMPA SIGNIFICATIVA
#
# Evitamos escolher diferenças de 1 ou 2 MW.
# ------------------------------------------------------------

MIN_RAMP = 40  # MW/h

increase = candidates[
    candidates["delta_load_1h"] >= MIN_RAMP
].copy()

decrease = candidates[
    candidates["delta_load_1h"] <= -MIN_RAMP
].copy()


# ------------------------------------------------------------
# 3. AGRUPAR CARGA EM FAIXAS DE 10 MW
#
# Isso permite encontrar observações com carga muito semelhante.
# ------------------------------------------------------------

LOAD_BIN = 10

increase["load_bin"] = (
    increase["load_current_mw"] / LOAD_BIN
).round() * LOAD_BIN

decrease["load_bin"] = (
    decrease["load_current_mw"] / LOAD_BIN
).round() * LOAD_BIN


# ------------------------------------------------------------
# 4. LOCALIZAR FAIXAS PRESENTES NOS DOIS SENTIDOS
# ------------------------------------------------------------

common_bins = sorted(
    set(increase["load_bin"])
    &
    set(decrease["load_bin"])
)

if not common_bins:
    raise ValueError(
        "Não foram encontrados pares de carga semelhantes "
        "com rampas opostas."
    )


# ------------------------------------------------------------
# 5. PROCURAR O PAR MAIS EXPRESSIVO
#
# Score:
#   - cargas o mais próximas possível;
#   - rampas com grande magnitude.
# ------------------------------------------------------------

pairs = []

for load_bin in common_bins:

    ups = increase[
        increase["load_bin"] == load_bin
    ]

    downs = decrease[
        decrease["load_bin"] == load_bin
    ]

    for _, up in ups.iterrows():
        for _, down in downs.iterrows():

            load_difference = abs(
                up["load_current_mw"]
                -
                down["load_current_mw"]
            )

            ramp_strength = (
                abs(up["delta_load_1h"])
                +
                abs(down["delta_load_1h"])
            )

            # Quanto menor a diferença de carga,
            # e maiores as rampas, melhor.
            score = (
                ramp_strength
                -
                5 * load_difference
            )

            pairs.append(
                (
                    score,
                    load_difference,
                    up,
                    down
                )
            )


pairs.sort(
    key=lambda x: x[0],
    reverse=True
)

_, load_difference, up_point, down_point = pairs[0]


_project_print("Par selecionado:")
_project_print()
_project_print("TRAJETÓRIA A — aumento")
_project_print(
    up_point[
        [
            "timestamp",
            "load_current_mw",
            "load_lag_1h",
            "delta_load_1h"
        ]
    ]
)

_project_print()
_project_print("TRAJETÓRIA B — redução")
_project_print(
    down_point[
        [
            "timestamp",
            "load_current_mw",
            "load_lag_1h",
            "delta_load_1h"
        ]
    ]
)

_project_print()
_project_print(
    f"Diferença entre cargas atuais: "
    f"{load_difference:.1f} MW"
)


# ------------------------------------------------------------
# 6. EXTRAIR JANELAS EM TORNO DOS DOIS PONTOS
#
# 4 horas antes e 2 horas depois.
# ------------------------------------------------------------

def get_context_window(center_time):

    return df_context[
        (
            df_context["timestamp"]
            >= center_time - pd.Timedelta(hours=4)
        )
        &
        (
            df_context["timestamp"]
            <= center_time + pd.Timedelta(hours=2)
        )
    ].copy()


window_up = get_context_window(
    up_point["timestamp"]
)

window_down = get_context_window(
    down_point["timestamp"]
)


# ------------------------------------------------------------
# 7. CRIAR FIGURA
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 6),
    sharey=True
)

fig.patch.set_facecolor(COLOR_BG)


# ------------------------------------------------------------
# PAINEL A — AUMENTANDO CARGA
# ------------------------------------------------------------

ax = axes[0]

ax.plot(
    window_up["timestamp"],
    window_up["load_current_mw"],
    marker="o",
    markersize=5,
    color=COLOR_ORANGE,
    linewidth=2.5
)

ax.scatter(
    up_point["timestamp"],
    up_point["load_current_mw"],
    s=90,
    color=COLOR_RED,
    zorder=5
)

ax.set_title(
    "Trajetória A · aumentando carga",
    loc="left",
    fontsize=14,
    color=COLOR_ORANGE,
    fontweight="bold"
)

ax.set_ylabel("Carga [MW]")


# ------------------------------------------------------------
# PAINEL B — REDUZINDO CARGA
# ------------------------------------------------------------

ax = axes[1]

ax.plot(
    window_down["timestamp"],
    window_down["load_current_mw"],
    marker="o",
    markersize=5,
    color=COLOR_RED,
    linewidth=2.5
)

ax.scatter(
    down_point["timestamp"],
    down_point["load_current_mw"],
    s=90,
    color=COLOR_RED,
    zorder=5
)

ax.set_title(
    "Trajetória B · reduzindo carga",
    loc="left",
    fontsize=14,
    color=COLOR_RED,
    fontweight="bold"
)


# ------------------------------------------------------------
# 8. ANOTAÇÕES DOS DOIS PONTOS
# ------------------------------------------------------------

axes[0].annotate(
    (
        f"P(t) = {up_point['load_current_mw']:.0f} MW\n"
        f"P(t−1) = {up_point['load_lag_1h']:.0f} MW\n"
        f"ΔP = +{up_point['delta_load_1h']:.0f} MW/h"
    ),
    xy=(
        up_point["timestamp"],
        up_point["load_current_mw"]
    ),
    xytext=(15, -60),
    textcoords="offset points",
    fontsize=10,
    color=COLOR_TEXT,
    arrowprops={
        "arrowstyle": "-",
        "color": COLOR_RED
    }
)

axes[1].annotate(
    (
        f"P(t) = {down_point['load_current_mw']:.0f} MW\n"
        f"P(t−1) = {down_point['load_lag_1h']:.0f} MW\n"
        f"ΔP = {down_point['delta_load_1h']:.0f} MW/h"
    ),
    xy=(
        down_point["timestamp"],
        down_point["load_current_mw"]
    ),
    xytext=(15, -60),
    textcoords="offset points",
    fontsize=10,
    color=COLOR_TEXT,
    arrowprops={
        "arrowstyle": "-",
        "color": COLOR_RED
    }
)


# ------------------------------------------------------------
# 9. FORMATAR DATAS
# ------------------------------------------------------------

for ax in axes:

    ax.xaxis.set_major_locator(
        mdates.HourLocator(interval=1)
    )

    ax.xaxis.set_major_formatter(
        mdates.DateFormatter("%Hh")
    )

    ax.set_xlabel("Tempo")


# ------------------------------------------------------------
# 10. TÍTULO GERAL
# ------------------------------------------------------------

fig.suptitle(
    "Mesma carga aproximada, contextos operacionais diferentes",
    x=0.07,
    ha="left",
    fontsize=18,
    color=COLOR_TEXT
)

fig.text(
    0.07,
    0.02,
    (
        "Dados reais da Iatan Unit 2 · "
        "os pontos destacados possuem carga semelhante, "
        "mas ΔP com sinais opostos."
    ),
    fontsize=9,
    color=COLOR_MUTED
)

plt.tight_layout(
    rect=[0, 0.05, 1, 0.93]
)

save_figure(
    fig,
    "03_mesma_carga_trajetorias_opostas"
)

plt.show()

In [ ]:
# ============================================================
# FIGURA 4
# STARTUP REAL — POR QUE A TRAJETÓRIA IMPORTA
# ============================================================
#
# OBJETIVO:
# Gerar a figura do slide 12 com dados reais da Iatan Unit 2,
# mostrando que, durante o startup, a unidade pode apresentar
# emissão positiva de CO2 antes de aparecer carga elétrica > 0.
#
# A figura é salva em:
#   outputs/figures/04_startup_real_trajetoria_importa.(png/svg)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. GARANTIR ORDENAÇÃO TEMPORAL
# ------------------------------------------------------------

df_plot = (
    df
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. IDENTIFICAR O PRIMEIRO STARTUP REAL
# ------------------------------------------------------------
#
# O startup foi definido no notebook por transição OFF -> ON.
# ------------------------------------------------------------

startup_rows = df_plot[
    df_plot["startup_event"] == True
].copy()

if startup_rows.empty:
    raise ValueError("Nenhum startup_event foi encontrado na base.")

startup_time = startup_rows.iloc[0]["timestamp"]

_project_print("Primeiro startup encontrado em:", startup_time)


# ------------------------------------------------------------
# 3. DEFINIR JANELA DO GRÁFICO
# ------------------------------------------------------------
#
# Vamos pegar:
#   - 3 horas antes
#   - 8 horas depois
#
# Assim o slide mostra claramente:
#   OFF -> startup/aquecimento -> carga aparecendo
# ------------------------------------------------------------

window_start = startup_time - pd.Timedelta(hours=3)
window_end   = startup_time + pd.Timedelta(hours=8)

window = df_plot[
    (df_plot["timestamp"] >= window_start)
    &
    (df_plot["timestamp"] <= window_end)
].copy()

if window.empty:
    raise ValueError("Janela do startup ficou vazia.")


# ------------------------------------------------------------
# 4. IDENTIFICAR PRIMEIRA HORA COM CARGA POSITIVA
# ------------------------------------------------------------

positive_load_rows = window[
    window["load_current_mw"] > 0
].copy()

if positive_load_rows.empty:
    first_positive_load_time = startup_time
else:
    first_positive_load_time = positive_load_rows.iloc[0]["timestamp"]

_project_print("Primeira carga > 0 em:", first_positive_load_time)


# ------------------------------------------------------------
# 5. VALORES-CHAVE
# ------------------------------------------------------------

row_startup = df_plot[
    df_plot["timestamp"] == startup_time
].iloc[0]


# ------------------------------------------------------------
# 6. IDENTIFICAR PERÍODO POSTERIOR AO STARTUP
#    COM OPERATING TIME = 1 E CARGA = 0
# ------------------------------------------------------------

mask_plateau = (
    (window["timestamp"] > startup_time)
    &
    (window["timestamp"] <= startup_time + pd.Timedelta(hours=5))
    &
    (window["Operating Time"] == 1)
    &
    (window["load_current_mw"] == 0)
)

plateau_rows = window[mask_plateau].copy()


# ------------------------------------------------------------
# 7. CRIAR FIGURA
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 7.5),
    sharex=True,
    gridspec_kw={
        "height_ratios": [1, 1],
        "hspace": 0.08
    }
)

fig.patch.set_facecolor(COLOR_BG)


# ============================================================
# PAINEL SUPERIOR — CARGA
# ============================================================

ax = axes[0]

ax.plot(
    window["timestamp"],
    window["load_current_mw"],
    color=COLOR_ORANGE,
    linewidth=2.8
)

# faixa sombreada do período de startup/aquecimento
ax.axvspan(
    startup_time,
    first_positive_load_time,
    color=COLOR_RED_LIGHT,
    alpha=0.55
)

# linha vertical indicando início do startup
ax.axvline(
    startup_time,
    color=COLOR_RED,
    linestyle="--",
    linewidth=1.4
)

ax.set_ylabel("Carga [MW]")

ax.set_title(
    "Comportamento real da unidade durante o startup",
    loc="left",
    fontsize=22,
    pad=15
)

ax.text(
    startup_time,
    ax.get_ylim()[1] * 0.96,
    "  início do startup",
    color=COLOR_RED,
    fontsize=10,
    va="top"
)

# texto sobre faixa sombreada
if first_positive_load_time > startup_time:

    middle_startup = startup_time + (
        first_positive_load_time - startup_time
    ) / 2

    ax.text(
        middle_startup,
        ax.get_ylim()[1] * 0.88,
        "startup / aquecimento",
        color=COLOR_RED,
        fontsize=11,
        ha="center"
    )


# ============================================================
# PAINEL INFERIOR — CO2
# ============================================================

ax2 = axes[1]

ax2.plot(
    window["timestamp"],
    window["CO2 Mass (short tons)"],
    color=COLOR_RED,
    linewidth=2.8
)

ax2.axvspan(
    startup_time,
    first_positive_load_time,
    color=COLOR_RED_LIGHT,
    alpha=0.55
)

ax2.axvline(
    startup_time,
    color=COLOR_RED,
    linestyle="--",
    linewidth=1.4
)

ax2.set_ylabel("CO₂ Mass [short tons]")
ax2.set_xlabel("Hora")


# ------------------------------------------------------------
# 8. FORMATAÇÃO DOS EIXOS
# ------------------------------------------------------------

for axx in axes:

    axx.spines["left"].set_color(COLOR_GRID)
    axx.spines["bottom"].set_color(COLOR_GRID)

    axx.tick_params(
        axis="both",
        colors=COLOR_MUTED
    )

    axx.margins(x=0.01)


axes[1].xaxis.set_major_locator(
    mdates.HourLocator(interval=1)
)

axes[1].xaxis.set_major_formatter(
    mdates.DateFormatter("%Hh")
)


# ------------------------------------------------------------
# 9. ANOTAÇÃO DO PONTO DE STARTUP — CARGA
# ------------------------------------------------------------

ax.annotate(
    (
        f"Operating Time = {row_startup['Operating Time']:.2f}\n"
        f"Carga = {row_startup['load_current_mw']:.0f} MW"
    ),
    xy=(
        row_startup["timestamp"],
        row_startup["load_current_mw"]
    ),
    xytext=(20, 40),
    textcoords="offset points",
    fontsize=10,
    color=COLOR_TEXT,
    arrowprops={
        "arrowstyle": "-",
        "color": COLOR_MUTED
    }
)


# ------------------------------------------------------------
# 10. ANOTAÇÃO DO PONTO DE STARTUP — CO2
# ------------------------------------------------------------

ax2.annotate(
    (
        f"CO₂ = "
        f"{row_startup['CO2 Mass (short tons)']:.3f} short tons"
    ),
    xy=(
        row_startup["timestamp"],
        row_startup["CO2 Mass (short tons)"]
    ),
    xytext=(20, 35),
    textcoords="offset points",
    fontsize=10,
    color=COLOR_TEXT,
    arrowprops={
        "arrowstyle": "-",
        "color": COLOR_MUTED
    }
)


# ------------------------------------------------------------
# 11. DESTACAR O INTERVALO EM QUE A CARGA AINDA É ZERO
#     MAS O CO2 CONTINUA POSITIVO
# ------------------------------------------------------------

if not plateau_rows.empty:

    first_plateau = plateau_rows.iloc[0]
    last_plateau = plateau_rows.iloc[-1]

    mid_time = (
        first_plateau["timestamp"]
        +
        (
            last_plateau["timestamp"]
            -
            first_plateau["timestamp"]
        ) / 2
    )

    ax.text(
        mid_time,
        ax.get_ylim()[1] * 0.12,
        "Carga ainda = 0 MW",
        fontsize=10,
        color=COLOR_ORANGE,
        ha="center"
    )

    ax2.text(
        mid_time,
        ax2.get_ylim()[1] * 0.78,
        "CO₂ permanece positivo",
        fontsize=10,
        color=COLOR_RED,
        ha="center"
    )


# ------------------------------------------------------------
# 12. RODAPÉ DISCRETO
# ------------------------------------------------------------

fig.text(
    0.08,
    0.01,
    (
        "Dados reais · Iatan Unit 2 · "
        "primeiro startup observado em 2025"
    ),
    fontsize=9,
    color=COLOR_MUTED
)


# ------------------------------------------------------------
# 13. AJUSTE FINAL
# ------------------------------------------------------------

plt.tight_layout(
    rect=[0, 0.03, 1, 1]
)


# ------------------------------------------------------------
# 14. SALVAR
# ------------------------------------------------------------

save_figure(
    fig,
    "04_startup_real_trajetoria_importa"
)

plt.show()

## Referências

CAWLEY, G. C.; TALBOT, N. L. C. On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation. Journal of Machine Learning Research, [s. l.], v. 11, n. 70, p. 2079-2107, 2010. Disponível em: https://jmlr.org/papers/v11/cawley10a.html. Acesso em: 29 out. 2026.

CERQUEIRA, V.; TORGO, L.; MOZETIČ, I. Evaluating time series forecasting models: an empirical study on performance estimation methods. Machine Learning, [s. l.], v. 109, n. 11, p. 1997-2028, 2020. DOI: https://doi.org/10.1007/s10994-020-05910-7. Disponível em: https://link.springer.com/article/10.1007/s10994-020-05910-7. Acesso em: 29 out. 2026.

GU, J.; SWARD, J. A.; ZHANG, K. M. Predicting power plant emissions using public data and machine learning. Environmental Science: Advances, [s. l.], v. 2, n. 12, p. 1696-1707, 2023. DOI: https://doi.org/10.1039/D3VA00191A. Disponível em: https://pubs.rsc.org/en/content/articlelanding/2023/va/d3va00191a. Acesso em: 29 out. 2026.

GUTHRIE, W. F. NIST/SEMATECH e-Handbook of Statistical Methods (NIST Handbook 151). Gaithersburg: National Institute of Standards and Technology, 2020. Seções 1.3.3.7, Box Plot, e 1.3.5.17, Detection of Outliers. DOI: https://doi.org/10.18434/M32189. Disponível em: https://www.itl.nist.gov/div898/handbook/eda/section3/boxplot.htm e https://www.itl.nist.gov/div898/handbook/eda/section3/eda35h.htm. Acesso em: 29 out. 2026.

JOLLIFFE, I. T.; CADIMA, J. Principal component analysis: a review and recent developments. Philosophical Transactions of the Royal Society A: Mathematical, Physical and Engineering Sciences, [s. l.], v. 374, n. 2065, art. 20150202, 2016. DOI: https://doi.org/10.1098/rsta.2015.0202. Disponível em: https://pmc.ncbi.nlm.nih.gov/articles/PMC4792409/. Acesso em: 29 out. 2026.

SCIKIT-LEARN. User Guide. Version 1.8.0. [S. l.]: The scikit-learn developers, [2025]. Seções MinMaxScaler e Regression metrics. Disponível em: https://scikit-learn.org/1.8/modules/generated/sklearn.preprocessing.MinMaxScaler.html e https://scikit-learn.org/1.8/modules/model_evaluation.html#regression-metrics. Acesso em: 29 out. 2026.

SURI, D.; DE CHALENDAR, J.; AZEVEDO, I. M. L. Assessing the real implications for CO₂ as generation from renewables increases. Nature Communications, [s. l.], v. 16, art. 7124, 2025. DOI: https://doi.org/10.1038/s41467-025-59800-4. Disponível em: https://www.nature.com/articles/s41467-025-59800-4. Acesso em: 29 out. 2026.

UNITED STATES. Environmental Protection Agency. Clean Air Markets Division. CAMD’s Power Sector Emissions Data Guide. [S. l.]: EPA, 18 jul. 2022. Disponível em: https://www.epa.gov/system/files/documents/2022-07/CAMD%27s%20Power%20Sector%20Emissions%20Data%20Guide%20-%2007182022.pdf. Acesso em: 29 out. 2026.


## Auditoria v2 — integridade, cortes, PCA e total anual

**Importante:** o valor de CO₂ original está em *short tons*. A comparação com a EPA só é válida se o escopo (facility/unidade), ano, versão e unidade de massa coincidirem. A soma das 8.760 horas tratadas é um indicador interno, não validação externa. O protocolo de cortes original é mantido para reprodutibilidade; outros notebooks podem adotar cortes diferentes, que devem ser explicitamente documentados.

In [ ]:
# V2.1 — Contrato de dados, conferência interna e comparação opcional com EPA
from pathlib import Path
import pandas as pd, numpy as np, json, hashlib
from IPython.display import display
AUDIT_V2 = Path(REPORTS) / "auditoria_02_v2_2025"
AUDIT_V2.mkdir(parents=True, exist_ok=True)
assert len(df) == 8760, f"Esperadas 8760 horas de 2025; obtidas {len(df)}"
assert pd.to_datetime(df['timestamp']).dt.year.eq(2025).all(), 'Há registros fora de 2025'
assert not df['timestamp'].duplicated().any(), 'Timestamps duplicados'
assert df['timestamp'].sort_values().diff().dropna().eq(pd.Timedelta(hours=1)).all(), 'Série não é horária contínua'
raw_short = pd.to_numeric(df['CO2 Mass (short tons)'], errors='coerce').sum(min_count=1)
treated_short = pd.to_numeric(df['co2_mass_short_tons_treated'], errors='coerce').sum(min_count=1)
short_to_metric = 0.90718474
annual = {'year':2025, 'hours':len(df), 'hours_on':int(df['is_on'].sum()),
          'co2_original_short_tons':float(raw_short), 'co2_treated_short_tons':float(treated_short),
          'co2_treated_metric_tonnes':float(treated_short*short_to_metric),
          'external_EPA_verification':'PENDING'}
# EPA CAM API Portal: referência anual oficial da mesma fonte CAMPD.
# O manifesto verifica a origem e a identidade da resposta preservada.
epa_dir = ROOT / "data" / "reference"
epa_record = epa_dir / "epa_proveniencia.json"
if epa_record.exists():
    meta = json.loads(epa_record.read_text(encoding="utf-8"))
    raw_epa = (epa_dir / "epa_anual_resposta.json").read_bytes()
    if hashlib.sha256(raw_epa).hexdigest() != meta["response_sha256"]:
        raise ValueError("Hash da resposta anual da EPA divergente.")
    matches = [
        row for row in json.loads(raw_epa)["items"]
        if str(row["facilityId"]) == str(selected_unit["facility_id"])
        and str(row["unitId"]) == str(selected_unit["unit_id"])
        and int(row["year"]) == int(selected_unit["year"])
    ]
    if len(matches) != 1:
        raise ValueError("A consulta anual não identifica univocamente a unidade e o ano.")
    official = float(matches[0]["co2Mass"])
    annual["epa_scope"] = f'{selected_unit["facility_id"]}/{selected_unit["unit_id"]}/{selected_unit["year"]}'
    annual["epa_official_short_tons"] = official
    annual["epa_difference_short_tons"] = float(treated_short - official)
    annual["epa_response_sha256"] = meta["response_sha256"]
    annual["epa_retrieved_utc"] = meta["retrieved_utc"]
    annual["external_EPA_verification"] = (
        "MATCH" if np.isclose(treated_short, official, rtol=0, atol=0.0005)
        else "DIFFERENCE"
    )
with open(AUDIT_V2/'annual_co2_audit.json','w',encoding='utf-8') as f: _project_dump(annual,f,indent=2,ensure_ascii=False)
_project_print('Auditoria anual:', annual['external_EPA_verification'])
_project_display(pd.DataFrame([annual]).T)


In [ ]:
# Jolliffe e Cadima (2016), PCA; Cawley e Talbot (2010), separação de treino e avaliação.
# V2.2 — Cortes históricos e PCA estritamente ajustada ao treino
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pandas as pd, numpy as np, json
splits = {'train':train_data,'validation':validation_data,'test':test_data}
rows=[]
for name,part in splits.items():
    rows.append({'split':name,'rows':len(part),'first':str(part['timestamp'].min()),
                 'last':str(part['timestamp'].max()),'year_2025_only':bool(part['timestamp'].dt.year.eq(2025).all())})
split_report=pd.DataFrame(rows)
assert all(split_report['year_2025_only'])
assert set(train_data.index).isdisjoint(validation_data.index)
assert set(train_data.index).isdisjoint(test_data.index)
assert set(validation_data.index).isdisjoint(test_data.index)
assert train_data['timestamp'].max() < validation_data['timestamp'].min()
assert validation_data['timestamp'].max() < test_data['timestamp'].min()
_project_csv(split_report, AUDIT_V2/'original_split_2025.csv',index=False)
# PCA exploratória adicional: treino apenas; avaliação transformada nas demais partições.
Xtr=train_data[model_features].replace([np.inf,-np.inf],np.nan)
imputer=SimpleImputer(strategy='median').fit(Xtr)
scaler=StandardScaler().fit(imputer.transform(Xtr))
pca_train=PCA().fit(scaler.transform(imputer.transform(Xtr)))
components=pd.DataFrame({'component':np.arange(1,len(pca_train.explained_variance_ratio_)+1),
 'variance_ratio_train':pca_train.explained_variance_ratio_,
 'cumulative_variance_train':np.cumsum(pca_train.explained_variance_ratio_)})
_project_csv(components, AUDIT_V2/'pca_train_only_variance.csv',index=False)
for name,part in splits.items():
    transformed=pca_train.transform(scaler.transform(imputer.transform(part[model_features].replace([np.inf,-np.inf],np.nan))))
    _project_csv(pd.DataFrame(transformed,columns=[f'PC{i+1}' for i in range(transformed.shape[1])],index=part.index), AUDIT_V2/f'pca_train_fitted_{name}.csv')
_project_print('Cortes originais preservados:'); _project_display(split_report)
_project_print('PCA treinada somente no treino:'); _project_display(components)


In [ ]:
# V2.3 — Dicionário, manifesto e checksum da base tratada
import json, hashlib, pandas as pd
p = DATA_INTERIM / 'step05_feature_engineering.parquet'
if not p.is_file():
    raise FileNotFoundError(p)
manifest={'version':'02_v2.0_audit','year':2025,'original_split':'jan-set / out / nov-dez',
 'split_policy':'Cortes históricos preservados; alinhar protocolos entre fases antes de seleção final',
 'epa_status':annual['external_EPA_verification'],
 'pca_policy':'PCA original exploratória; PCA adicional ajustada apenas no treino',
 'source_parquet':str(p),'source_parquet_sha256':hashlib.sha256(p.read_bytes()).hexdigest(),
 'output_folder':str(AUDIT_V2)}
with open(AUDIT_V2/'audit_manifest.json','w',encoding='utf-8') as f:_project_dump(manifest,f,indent=2,ensure_ascii=False)
dictionary=pd.DataFrame({'variable':df.columns,'dtype':[str(df[c].dtype) for c in df.columns],
 'missing_count':[int(df[c].isna().sum()) for c in df.columns]})
dictionary['unit_or_definition']='Consultar documentação e código original; não inferido automaticamente'
for c,u in {'timestamp':'data/hora','Gross Load (MW)':'MW','CO2 Mass (short tons)':'short tons',
 'co2_mass_short_tons_treated':'short tons','load_current_mw':'MW'}.items():
    dictionary.loc[dictionary.variable.eq(c),'unit_or_definition']=u
_project_csv(dictionary, AUDIT_V2/'data_dictionary_2025.csv',index=False)
_project_print('Auditoria concluída em:',AUDIT_V2)
_project_print('Comparação com o agregado anual da EPA:', annual['external_EPA_verification'])
